# BACE1 activity modeling — public code sample

Prior exploratory research notebook supplied by David Shen.

**Lab datasets and all saved research outputs are withheld.** This public copy
contains analysis code and explanatory text only. See `../README.md` and
`../data/README.md` for data restrictions, dependencies and execution limits.

This is a historical exploration with alternative input files and branches,
not a supported end-to-end Run All pipeline. Some selected feature identities
are intentionally replaced with `REDACTED_FEATURE_###` placeholders; these do
not name real columns and must not be mistaken for research results.

Reading path: molecular fingerprints and identifier alignment; target/data
preparation; baseline regression and visualization; repeated feature/model
comparisons; alternative selection and interpretation branches.

Public-export preparation cleared outputs and metadata, removed data-size
narration, generalized run labels and redacted hard-coded selected features.
The original model logic and historical APIs otherwise remain exploratory.
No private analysis or model accuracy was rerun or verified for this export.


**Molecular Fingerprinting of BACE1 Inhibition Data**

The fingerprint-preparation workflow references the Data Professor bioinformatics tutorial. Lab compound records are not distributed.


First, download the PaDEL-Descriptor.

In [ ]:
! wget https://github.com/dataprofessor/bioinformatics/raw/master/padel.zip
! wget https://github.com/dataprofessor/bioinformatics/raw/master/padel.sh

In [ ]:
! unzip padel.zip

Then load the BACE1 Inhibition Data file.

In [ ]:
import pandas as pd

In [ ]:
df3 = pd.read_csv('BACE1_PDB_all_data.csv')
df3['BindingDB_MonomerID'] = ['PDB' + str(i+1) for i in range(len(df3))]
df3.to_csv('BACE1_PDB_all_data_exd.csv')
print(df3.shape)
df3

In [ ]:
selection = ['SMILES','BindingDB_MonomerID']
df3_selection = df3[selection]
df3_selection.to_csv('molecule.smi', sep='\t', index=False, header=False)

In [ ]:
! cat molecule.smi | head -5

In [ ]:
! cat molecule.smi | wc -l

Now it's time to calculate fingerprint descriptors!
java -Xms1G -Xmx1G -Djava.awt.headless=true -jar ./PaDEL-Descriptor/PaDEL-Descriptor.jar -removesalt -standardizenitro -fingerprints -descriptortypes ./PaDEL-Descriptor/PubchemFingerprinter.xml -dir ./ -file descriptors_output.csv

First, calculate PaDEL descriptors.

In [ ]:
! cat padel.sh

In [ ]:
! bash padel.sh

In [ ]:
! ls -l

Descriptor generation can omit compounds. Match descriptor and activity records by identifier before constructing the modeling table. Private dataset counts are omitted from this public copy.


In [ ]:
pip install pandas

In [ ]:
import pandas as pd

def match_and_extract_components(file1, file2, output_file):
    # Read Excel files into Pandas DataFrames
    #df1 = pd.read_excel(file1)
    #df2 = pd.read_excel(file2)
    df1 = pd.read_csv(file1)
    df2 = pd.read_csv(file2)

    # Extract the "Component" column from each DataFrame
    components1 = set(df1['BindingDB_MonomerID'])
    components2 = set(df2['Name'])

    # Find common components
    common_components = components1.intersection(components2)

    # Filter rows in df1 based on common components
    matched_df1 = df1[df1['BindingDB_MonomerID'].isin(common_components)]

    # Save the result to a new Excel file
    #matched_df1.to_excel(output_file, index=False)
    matched_df1.to_csv(output_file, index=False)

    print(f"Matching components extracted to {output_file}")

# Replace 'file1.xlsx', 'file2.xlsx', and 'output_file.xlsx' with your actual file names
match_and_extract_components('BACE1_PDB_all_data_exd.csv', 'descriptors_output.csv', 'output_file.csv')


In [ ]:
test = pd.read_csv('output_file.csv')
test

Time to prepare the X and Y Data Matrices.

X data matrix.

In [ ]:
df3_X = pd.read_csv('descriptors_output.csv')
df3_X_ext = pd.read_csv('descriptors_output.csv')

In [ ]:
df3_X

In [ ]:
df3_X_ext

Y variable.

Convert IC50 to pIC50.

For IC50 measurements with confirmed nanomolar units, convert using `pIC50 = -log10(IC50_nM * 1e-9)`. Inputs must be positive. A later branch renames an existing mixed activity field; that rename does not by itself harmonize pKd, pKi and pIC50.


In [ ]:
import pandas as pd
import numpy as np

# Replace 'your_excel_file.xlsx' with the actual name of your Excel file
# Replace 'IC50_nM_.1' with the actual name of your column
# Replace 'output_file.xlsx' with the desired name for your new Excel file

# Read data from Excel file into a DataFrame
df = pd.read_csv('output_file_big.csv')

# Convert from nM to M, then apply -log10
df['pIC50'] = -np.log10(df['IC50_nM_.1'] * 1e-9)

# Drop the original 'IC50_nM_.1' column
df.drop('IC50_nM_.1', axis=1, inplace=True)

# Save the modified DataFrame to a new Excel file
df.to_csv('output_file2_big.csv', index=False)


In [ ]:
import pandas as pd
import numpy as np

# Read data from Excel file into a DataFrame
df = pd.read_csv('output_file.csv')
df.rename(columns={'pKd pKi pIC50': 'pIC50'}, inplace=True)
df

Add IC50 column into the file.

In [ ]:
selection1 = ['pIC50','BindingDB_MonomerID']
df3_Y = df[selection1]
df3_Y

df3_X.set_index('Name', inplace=True)
df3_Y.set_index('BindingDB_MonomerID', inplace=True)

Combining X and Y variable.

In [ ]:
dataset3 = pd.concat([df3_X,df3_Y], axis=1, join='inner')
dataset3.reset_index(inplace=True)
dataset3.rename(columns={'index': 'NAME'}, inplace=True)
dataset3 = dataset3.drop(columns=['NAME'])
dataset3

In [ ]:
dataset3.to_csv('output_file_qsar.csv', index=False)

In [ ]:
docking_ind_features = ['BindingDB_MonomerID','Hbond','Hphob','VwInt','Eintl','Dsolv','SolEl','dTSsc','pIC50']
df3_ext_Y = df[docking_ind_features]

df3_X_ext.set_index('Name', inplace=True)
df3_ext_Y.set_index('BindingDB_MonomerID', inplace=True)

In [ ]:
dataset3_ext = pd.concat([df3_X_ext,df3_ext_Y], axis=1)
dataset3_ext.reset_index(inplace=True)
dataset3_ext.rename(columns={'index': 'NAME'}, inplace=True)
dataset3_ext = dataset3_ext.drop(columns=['NAME'])
dataset3_ext

In [ ]:
dataset3_ext.to_csv('output_file_qsar-docking.csv', index=False)

Now download the csv file for the next part: Model Building.

**Time to build a regression model with RandomForestRegressor!**

Import libraries.

In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

Load the data set.

In [ ]:
df = pd.read_csv('output_file3.csv')

In [ ]:
df = pd.read_csv('output_file3_all.csv')
df.shape

Input features consist of molecular fingerprint descriptors; the target is named pIC50 in this legacy analysis.


In [ ]:
X = df.drop('pIC50', axis=1)
X

Output features.

In [ ]:
Y = df.pIC50
Y.shape

Examine the data dimension.

In [ ]:
X.shape

In [ ]:
Y.shape

Remove low variance features.

In [ ]:
from sklearn.feature_selection import VarianceThreshold
selection = VarianceThreshold(threshold=(.8 * (1 - .8)))
X_reduced = selection.fit_transform(X)

mask = selection.get_support()
remain_predictors = X.columns[mask]
threshold = (.8 * (1 - .8))

In [ ]:
threshold

In [ ]:
X_reduced.shape
print(X_reduced)

It is time to use Correlation Heatmap (Binary Predictors with pIC50) to draw a heat map

In [ ]:
!pip install matplotlib

Data split (50/50) ratio.

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.5)

In [ ]:
X_train.shape, Y_train.shape

In [ ]:
X_test.shape, Y_test.shape

Build a regression model using Random Forest.

In [ ]:
import numpy as np
np.random.seed(100)
model = RandomForestRegressor(n_estimators=100)
model.fit(X_train, Y_train)
r2 = model.score(X_test, Y_test)
r2

In [ ]:
Y_pred = model.predict(X_test)

Make scatter plot of Experimental vs. Predicted pIC50 values.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(color_codes=True)
sns.set_style("white")

ax = sns.regplot(x=Y_test, y=Y_pred, scatter_kws={'alpha':0.4})
ax.set_xlabel('Experimental pIC50', fontsize='large', fontweight='bold')
ax.set_ylabel('Predicted pIC50', fontsize='large', fontweight='bold')
ax.set_xlim(0, 12)
ax.set_ylim(0, 12)
ax.figure.set_size_inches(5, 5)
plt.show

Visualization of QSAR data set

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

dfv = pd.read_csv('output_file3.csv') # read in QSAR only dataset
dfv_sorted = dfv.sort_values(by='pIC50', ascending=False)

dfv_selected = dfv_sorted[remain_predictors.union(['pIC50'])]
Xv = dfv_selected.iloc[:, :-1]
sorted_pIC50 = dfv_sorted['pIC50']
print(Xv.shape)

# Step 3: Normalize pIC50 values for the color bar
norm = plt.Normalize(sorted_pIC50.min(), sorted_pIC50.max())
sm = plt.cm.ScalarMappable(cmap="Blues", norm=norm)
sm.set_array([])

# Step 4: Set figure size to make cells square
plt.figure(figsize=(len(Xv.columns) * 0.2, len(Xv) * 0.2))

ax = sns.heatmap(Xv, cmap=sns.color_palette(["yellow", "red"]), cbar=False, linewidths=0.01, linecolor='white', yticklabels=False, square=True)
plt.title('Heatmap of Binary Predictors Sorted by pIC50', fontsize=18)
plt.xlabel('Predictors', fontsize=14)

# Step 6: Add a vertical color bar for pIC50 values
plt.gcf().subplots_adjust(left=0.25, right=0.85)
cbar_ax = plt.gcf().add_axes([0.21, 0.2025, 0.02, 0.6])  # Position the color bar
plt.colorbar(sm, cax=cbar_ax)
# Step 7: Position the ylabel on the left of the color bar
cbar_ax.yaxis.set_label_position('left')
cbar_ax.set_ylabel('pIC50', fontsize=14, labelpad=20, rotation=270)

plt.savefig('qsar_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

OK, lets try Clustered Heatmap

In [ ]:
#import seaborn as sns
#import matplotlib.pyplot as plt

plt.figure(figsize=(12, 12))  # Adjust figure size
sns.clustermap(Xv, cmap=sns.color_palette(["black", "#00FF00"], as_cmap=True), linewidths=0, yticklabels=False, col_cluster=True, row_cluster=False)
plt.title('Clustered Heatmap of Binary Predictors')
plt.show()

In [ ]:
# Correlation Matrix Plot

corr = Xv.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap='coolwarm', annot=False, linewidths=0.5)
plt.title('Correlation Matrix of Binary Predictors')
plt.show()

In [ ]:
# Principal Component Analysis (PCA) Plot
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2)
principalComponents = pca.fit_transform(Xv)

plt.figure(figsize=(10, 8))
plt.scatter(principalComponents[:, 0], principalComponents[:, 1], c=sorted_pIC50, cmap='Reds')
plt.title('PCA of Binary Predictors')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.colorbar(label='pIC50')
plt.show()

In [ ]:
# Barplot for Proportion of 1s Across Predictors
proportions = Xv.mean()

plt.figure(figsize=(16, 6))
proportions.sort_values().plot(kind='bar', color='skyblue')
plt.title('Proportion of 1s in Each Binary Predictor')
plt.xlabel('Predictors')
plt.ylabel('Proportion of 1s')
plt.show()


In [ ]:
#Lets try bipartite Graph
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd

# Step 1: Load your data
# Assuming your dataset is in a DataFrame called df, where the last column is the pIC50 target
# and the first 156 columns are the binary predictors.
dfn = dfv_selected

# Sample structure for illustrative purposes
# df = pd.DataFrame({
#     'predictor1': [0, 1, 0, 1],
#     'predictor2': [1, 0, 1, 1],
#     ...
#     'pIC50': [5.6, 7.8, 6.1, 4.5]
# })

# Step 2: Binarize the pIC50 if necessary (or create categories)
# You can categorize pIC50 values, but if you want to keep them as continuous values, you can skip this step.
#dfn['pIC50_category'] = pd.qcut(dfn['pIC50'], q=4, labels=['low', 'medium-low', 'medium-high', 'high'])

# Step 3: Create the bipartite graph
B = nx.Graph()

# Add pIC50 nodes
pIC50_nodes = dfn['pIC50'].unique()  # You can use categorized pIC50 if you binned it
B.add_nodes_from(pIC50_nodes, bipartite=0)

# Add predictor nodes
predictors = dfn.columns[:-1]  # All columns except the last one
B.add_nodes_from(predictors, bipartite=1)


# Step 4: Add edges between predictors and pIC50 values
for index, row in dfn.iterrows():
    for predictor in predictors:
        if row[predictor] == 1:
            B.add_edge(row['pIC50'], predictor)  # Use row['pIC50_category'] if categorized

# Step 5: Draw the bipartite graph
plt.figure(figsize=(14, 10))

# Define the layout for bipartite graphs
pos = nx.bipartite_layout(B, pIC50_nodes)

# Draw nodes and edges with colors
nx.draw_networkx_nodes(B, pos, nodelist=pIC50_nodes, node_color='lightgreen', node_size=500, alpha=0.8, label='pIC50 Categories')
nx.draw_networkx_nodes(B, pos, nodelist=predictors, node_color='lightblue', node_size=500, alpha=0.8, label='Predictors')
nx.draw_networkx_edges(B, pos, width=1.0, alpha=0.5)
nx.draw_networkx_labels(B, pos, font_size=10)

plt.title("Bipartite Graph of Predictors and pIC50")
plt.legend()
plt.show()

In [ ]:
import csv

# Extract edges from the bipartite graph
edges = list(B.edges())
print(edges)

modified_edges = [(edge[0], edge[1], 1, 1) for edge in edges]

# Specify the output CSV file name
output_file = 'bipartite_graph.csv'

# Write the edges to a CSV file
with open(output_file, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['Source', 'Target'])  # Writing header
    for edge in modified_edges:
        writer.writerow(edge)

print(f"Graph data has been saved to {output_file}")

In [ ]:
import pandas as pd

# Load the CSV file into a DataFrame
dfc = pd.read_csv('bipartite_graph.csv')
print(dfc.shape)

# Generate the desired list of lists
result = dfc.values.tolist()

# Print the result
print(result)

In [ ]:
dfv_selected

Comparing Regressors!


Import libraries.

In [ ]:
! pip install lazypredict

Regressor comparisons and repeated-split experiments.


In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
import lazypredict
from lazypredict.Supervised import LazyRegressor

Load the data set.

In [ ]:
docking_features = ['Hbond','Hphob','VwInt','Eintl','Dsolv','SolEl','dTSsc', 'pIC50']
df_Q = pd.read_csv('output_file_qsar.csv')
df_QD = pd.read_csv('output_file_qsar-docking.csv')
df_D = df_QD[docking_features]
print(df_Q.shape)
print(df_QD.shape)
print(df_D.shape)

In [ ]:
X = df_QD.drop('pIC50', axis=1)
Y = df_QD.pIC50
X

Data pre-processing.

In [ ]:
# Examine X dimension
X.shape

In [ ]:
# Remove low variance features
from sklearn.feature_selection import VarianceThreshold
selection = VarianceThreshold(threshold=(.8 * (1 - .8)))
X_reduced = selection.fit_transform(X)
retained_columns = X.columns[selection.get_support(indices=True)]
X_reduced = pd.DataFrame(X_reduced, columns=retained_columns)
print(X_reduced.shape)
#scaler = StandardScaler()
#X_scaled = scaler.fit_transform(X_reduced)
#X_scaled = pd.DataFrame(X_scaled, columns=retained_columns)

In [ ]:
feature_names = X_reduced.columns
print(feature_names)

Split Training and Testing data 80% vs 20%
Training set vs Testing set
==================================================

In [ ]:
import os
from datetime import datetime

def saveSplitstoFiles(X_train, X_test, Y_train, Y_test, current_time, subdirectory, i, prefix=None):

    # Save Split to files
    # Specify the subdirectory
    ##subdirectory = 'Splits'

    # Get the current time and format it as a string
    ##current_time = datetime.now().strftime('%m-%d_%H-%M-%S')

    # Combine the subdirectory and file name into a full path
    #full_path = os.path.join(subdirectory, filename)

    # Create the subdirectory if it doesn't exist
    if prefix is None:
        prefix = ""
    os.makedirs(subdirectory, exist_ok=True)
    # Create the file name with the current time
    filename = f'{prefix}_XTrain_{current_time}_{i}.csv'
    X_train.to_csv(os.path.join(subdirectory, filename), index=True)
    filename = f'{prefix}_XTest_{current_time}_{i}.csv'
    X_test.to_csv(os.path.join(subdirectory, filename), index=True)
    filename = f'{prefix}_YTrain_{current_time}_{i}.csv'
    Y_train.to_csv(os.path.join(subdirectory, filename), index=True)
    filename = f'{prefix}_YTest_{current_time}_{i}.csv'
    Y_test.to_csv(os.path.join(subdirectory, filename), index=True)

In [ ]:
# Good method Ridge regression + cross validation
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeCV, Ridge
from sklearn.model_selection import cross_val_score

def featureSelectionCoe(Xf_train, Yf_train, current_time, subdirectory, num, i=None):
    # Step 1: Standardize the features

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(Xf_train)

    ##print("Are there NaN values in X_scaled?:", np.isnan(X_scaled).any())
    ##print("Number of NaN values in each column:\n", np.isnan(X_scaled).sum(axis=0))
    X_scaled_clean = X_scaled[~np.isnan(X_scaled).any(axis=1)]
    Y_clean = Yf_train[~np.isnan(X_scaled).any(axis=1)]

    # Step 2: Use RidgeCV to automatically find the best alpha
    alphas = np.logspace(-6, 6, 13)  # Range of alpha values to test
    #alphas = np.logspace(np.log10(best_alpha) - 2, np.log10(best_alpha) + 2, 15)
    ridge_cv = RidgeCV(alphas=alphas, cv=5, scoring='neg_mean_squared_error')
    ridge_cv.fit(X_scaled_clean, Y_clean)

    # Step 3: Fit the Ridge model with the optimal alpha
    best_alpha = ridge_cv.alpha_
    ridge = Ridge(alpha=best_alpha, max_iter=10000, random_state=42)
    ridge.fit(X_scaled_clean, Y_clean)

    # Step 3: Get the absolute values of the coefficients
    coef_abs = np.abs(ridge.coef_)

    # Step 4: Rank features based on the magnitude of their coefficients
    coef_ranking = pd.Series(coef_abs, index=Xf_train.columns).sort_values(ascending=False)

    # Step 5: Select top N features
    N = num  # Example: Select the top 50 features
    selected_features_ridge = coef_ranking.head(N).index
    ##print(selected_features_ridge)
    #cmb_selected_features = selected_features.union(additional_predictors)
    # Convert selected features to a DataFrame
    df_selected_features = pd.DataFrame(selected_features_ridge, columns=['Selected Features'])

    # Save to CSV
    ##filename = f'selected_features_{current_time}{i}.csv'
    ##df_selected_features.to_csv(os.path.join(subdirectory, filename), index=False)
    return df_selected_features;

In [ ]:
def selectTopFeatures(X_train, X_test, Y_train, Y_test):
    all_selected_features = set()
    selection_runs = []
    for i in range(50):
        # Perform data splitting using 80/20 ratio


        docking_features_only = ['Hbond','Hphob','VwInt','Eintl','Dsolv','SolEl','dTSsc']
        XQ_train = X_train.drop(columns=docking_features_only)
        XQ_test = X_test.drop(columns=docking_features_only)

        Xf_train, Xf_test, Yf_train, Yf_test = train_test_split(XQ_train, Y_train, test_size=0.2, random_state=None)

        #saveSplitstoFiles(Xf_train, Xf_test, Yf_train, Yf_test, current_time, subdirectory, i, "f")

        selected_fs = featureSelectionCoe(Xf_train, Yf_train, current_time, subdirectory, 12)
        s = selected_fs.iloc[:, 0].tolist()

        selection_runs.append(s)
        all_selected_features.update(s)
    
    all_features_list = list(all_selected_features)
    features_df = pd.DataFrame(0, index=range(50), columns=all_features_list)

    # Step 4: Populate the DataFrame
    for i, selected_features in enumerate(selection_runs):
        features_df.loc[i, selected_features] = 1  # Set selected features to 1 for the i-th run

    # View the final DataFrame
    #print(features_df)
    #filename = f'allSfeatures_{current_time}.csv'
    #features_df.to_csv(os.path.join(subdirectory, filename), index=True)

    feature_sums = features_df.sum(axis=0)
    sorted_features = feature_sums.sort_values(ascending=False)
    top_12_features = sorted_features.head(10)
    return top_12_features.index.tolist()


In [ ]:
import numpy as np
import csv

current_time = datetime.now().strftime('%m-%d_%H-%M-%S')
subdirectory = 'Splits'
all_top_selected_features = set()
top_selection_runs = []
lazyResults_Qdf = pd.DataFrame()
lazyResults_Ddf = pd.DataFrame()
lazyResults_Cdf = pd.DataFrame()

for i in range(50) :
    
    #independent features selection:
    X_train, X_test, Y_train, Y_test = train_test_split(X_reduced, Y, test_size=0.2, random_state=None)
    docking_features_only = ['Hbond','Hphob','VwInt','Eintl','Dsolv','SolEl','dTSsc']
    XQ_train = X_train.drop(columns=docking_features_only)
    XQ_test = X_test.drop(columns=docking_features_only)

    saveSplitstoFiles(X_train, X_test, Y_train, Y_test, current_time, subdirectory, i)
    top_fs = selectTopFeatures(X_train, X_test, Y_train, Y_test)
    #ts = top_fs.iloc[:, 0].tolist()
    top_selection_runs.append(top_fs)
    all_top_selected_features.update(top_fs)
    
    XD_train = X_train[docking_features_only]
    XD_test = X_test[docking_features_only]
    YD_train = Y_train
    YD_test = Y_test
    clf = LazyRegressor(verbose=0,ignore_warnings=True, custom_metric=None)
    models_train,predictionsD_train = clf.fit(XD_train, XD_train, YD_train, YD_train)
    models_test,predictionsD_test = clf.fit(XD_train, XD_test, YD_train, YD_test)

    predictionsD_test_file = predictionsD_test.reset_index()
    filename = f'predictionsD_test_{current_time}{i}.csv'
    predictionsD_test_file.to_csv(os.path.join(subdirectory, filename), index=False)

    predictionsD_train_aligned = predictionsD_train.reindex(predictionsD_test.index)
    predictionsD_train_file = predictionsD_train_aligned.reset_index()
    filename = f'predictionsD_train_{current_time}{i}.csv'
    predictionsD_train_file.to_csv(os.path.join(subdirectory, filename), index=False)

    XQS_train = XQ_train[top_fs]
    #print(XQS_train)
    XQS_test = XQ_test[top_fs]
    clf = LazyRegressor(verbose=0,ignore_warnings=True, custom_metric=None)
    models_train,predictionsQ_train = clf.fit(XQS_train, XQS_train, Y_train, Y_train)
    models_test,predictionsQ_test = clf.fit(XQS_train, XQS_test, Y_train, Y_test)

    predictionsQ_test_file = predictionsQ_test.reset_index()
    filename = f'predictionsQ_test_{current_time}{i}.csv'
    predictionsQ_test_file.to_csv(os.path.join(subdirectory, filename), index=False)

    predictionsQ_train_aligned = predictionsQ_train.reindex(predictionsQ_test.index)
    predictionsQ_train_file = predictionsQ_train_aligned.reset_index()
    filename = f'predictionsQ_train_{current_time}{i}.csv'
    predictionsQ_train_file.to_csv(os.path.join(subdirectory, filename), index=False)


    docking_features_only_list = list(docking_features_only)
    cmb_features = top_fs + docking_features_only_list
    XC_train = X_train[cmb_features]
    XC_test = X_test[cmb_features]
    clf = LazyRegressor(verbose=0,ignore_warnings=True, custom_metric=None)
    models_train,predictionsC_train = clf.fit(XC_train, XC_train, Y_train, Y_train)
    models_test,predictionsC_test = clf.fit(XC_train, XC_test, Y_train, Y_test)

    predictionsC_test_file = predictionsC_test.reset_index()
    filename = f'predictionsC_test_{current_time}{i}.csv'
    predictionsC_test_file.to_csv(os.path.join(subdirectory, filename), index=False)

    predictionsC_train_aligned = predictionsC_train.reindex(predictionsC_test.index)
    predictionsC_train_file = predictionsC_train_aligned.reset_index()
    filename = f'predictionsC_train_{current_time}{i}.csv'
    predictionsC_train_file.to_csv(os.path.join(subdirectory, filename), index=False)
    
    r_squared_values_q = predictionsQ_test["R-Squared"]
    r_squared_values_d = predictionsD_test["R-Squared"]
    r_squared_values_c = predictionsC_test["R-Squared"]
    
    if i == 0:
        lazyResults_Qdf = pd.DataFrame(index=predictionsQ_test.index)
        lazyResults_Ddf = pd.DataFrame(index=predictionsD_test.index)
        lazyResults_Cdf = pd.DataFrame(index=predictionsC_test.index)
    lazyResults_Qdf[f'r{i+1}'] = r_squared_values_q.reindex(lazyResults_Qdf.index)
    lazyResults_Ddf[f'r{i+1}'] = r_squared_values_d.reindex(lazyResults_Ddf.index)
    lazyResults_Cdf[f'r{i+1}'] = r_squared_values_c.reindex(lazyResults_Cdf.index)

#print(lazyResults_df)
filename = f'allQResults_{current_time}.csv'
lazyResults_Qdf.to_csv(os.path.join(subdirectory, filename), index=True)
filename = f'allDResults_{current_time}.csv'
lazyResults_Ddf.to_csv(os.path.join(subdirectory, filename), index=True)
filename = f'allCResults_{current_time}.csv'
lazyResults_Cdf.to_csv(os.path.join(subdirectory, filename), index=True)

filename = f'allTopFeatures_{current_time}.csv'
# Writing list of lists to a CSV file
with open(filename, mode='w', newline='') as file:
    writer = csv.writer(file)
    # Write each inner list as a row in the CSV file
    writer.writerows(top_selection_runs)
    
all_top_features_list = list(all_top_selected_features)
top_features_df = pd.DataFrame(0, index=range(50), columns=all_top_features_list)
# Step 4: Populate the DataFrame
for i, selected_features in enumerate(top_selection_runs):
    top_features_df.loc[i, selected_features] = 1  # Set selected features to 1 for the i-th run

# View the final DataFrame
#print(features_df)
#filename = f'allSfeatures_{current_time}.csv'
#features_df.to_csv(os.path.join(subdirectory, filename), index=True)

top_feature_sums = top_features_df.sum(axis=0)
top_sorted_features = top_feature_sums.sort_values(ascending=False)
print(top_sorted_features)

In [ ]:
top_sorted_features_df = top_features_df[top_sorted_features.index]
print(top_sorted_features_df)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Define a colormap
cmap = sns.color_palette("coolwarm", as_cmap=True)
plt.figure(figsize=(12, 12))  # Adjust figure size
#sns.clustermap(sorted_features_df, cmap=sns.color_palette(["black", "#00FF00"], as_cmap=True), linewidths=0, yticklabels=False, col_cluster=False, row_cluster=True)
sns.clustermap(
    top_sorted_features_df,
    cmap=cmap,
    annot=False,
    cbar=True,
    linewidths=0.5,
    linecolor='gray',
    figsize=(10, 8),  # Adjust the size of the heatmap
    method='average',  # Clustering method (e.g., 'single', 'complete', 'average', etc.)
    metric='euclidean',  # Distance metric (e.g., 'euclidean', 'manhattan', etc.)
    row_cluster=True,  # Perform clustering on rows
    col_cluster=False  # Do not perform clustering on columns (only on rows)
)
plt.savefig('features.pdf')
plt.show()

In [ ]:
pip install --upgrade seaborn pandas

In [ ]:
# Function to plot KDE using matplotlib
def plot_kde(data, color='blue'):
    # Remove NaN and infinite values
    data = np.array(data)
    data = data[~np.isnan(data)]  # Remove NaN
    data = data[np.isfinite(data)]  # Remove infinite values

    # Define the bandwidth for KDE (this can be tuned)
    bandwidth = 0.5

    # Create the KDE model
    kde = KernelDensity(bandwidth=bandwidth, kernel='gaussian')
    kde.fit(data[:, None])

    # Generate points over the range of the data
    x_vals = np.linspace(min(data), max(data), 1000)
    
    # Compute the log density (convert back to density)
    log_dens = kde.score_samples(x_vals[:, None])
    dens = np.exp(log_dens)

    # Plot the KDE
    plt.plot(x_vals, dens, color=color, linewidth=2)


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KernelDensity


# Determine the number of rows
num_rows = lazyResults_Qdf.shape[0]

# Calculate the number of columns and rows needed for subplots
cols = 3  # Number of columns you want in the grid (adjustable)
rows = math.ceil(num_rows / cols)  # Calculate the number of rows needed

# Create a figure to hold all subplots
plt.figure(figsize=(15, rows * 3))  # Adjust the size of the figure

# Loop through all rows and plot each as a histogram
for i, row in enumerate(lazyResults_Qdf.index):
    plt.subplot(rows, cols, i + 1)
    
    # Get the data for the current row
    data = lazyResults_Qdf.loc[row]
    
    # Calculate mean and standard deviation
    mean = data.mean()
    std_dev = data.std()
    
    plot_kde(data, color='blue')
    
    # Set the title with mean and standard deviation
    plt.title(f'Histogram for {row}\nMean = {mean:.2f} ± {std_dev:.2f}')
    
    # Set x and y labels
    plt.xlabel('R-squared Values')
    plt.ylabel('Frequency')

# Adjust layout to prevent overlap
plt.tight_layout()

# Save the plot to a PDF file
plt.savefig('Qplotwlines.pdf')

# Show the plot
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import math

# Determine the number of rows
num_rows = lazyResults_Qdf.shape[0]

# Calculate the number of columns and rows needed for subplots
cols = 3  # Number of columns you want in the grid (adjustable)
rows = math.ceil(num_rows / cols)  # Calculate the number of rows needed

# Create a figure to hold all subplots
plt.figure(figsize=(15, rows * 3))  # Adjust the size of the figure

# Loop through all rows and plot each as a histogram
for i, row in enumerate(lazyResults_Qdf.index):
    plt.subplot(rows, cols, i + 1)
    data = lazyResults_Qdf.loc[row]
    
    mean = data.mean()
    std_dev = data.std()
    
    plt.hist(data, bins=50, color='blue', edgecolor='black')
    plt.title(f'Histogram for {row}\nMean = {mean:.2f} ± {std_dev:.2f}')
    plt.xlabel('R-squared Values')
    plt.ylabel('Frequency')

# Adjust layout to prevent overlap
plt.tight_layout()
plt.savefig('Q_plot.pdf')
# Show the plot
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import math

# Determine the number of rows
num_rows = lazyResults_Ddf.shape[0]

# Calculate the number of columns and rows needed for subplots
cols = 3  # Number of columns you want in the grid (adjustable)
rows = math.ceil(num_rows / cols)  # Calculate the number of rows needed

# Create a figure to hold all subplots
plt.figure(figsize=(15, rows * 3))  # Adjust the size of the figure

# Loop through all rows and plot each as a histogram
for i, row in enumerate(lazyResults_Ddf.index):
    plt.subplot(rows, cols, i + 1)
    data = lazyResults_Ddf.loc[row]
    
    mean = data.mean()
    std_dev = data.std()
    
    plt.hist(data, bins=50, color='blue', edgecolor='black')
    plt.title(f'Histogram for {row}\nMean = {mean:.2f} ± {std_dev:.2f}')
    plt.xlabel('R-squared Values')
    plt.ylabel('Frequency')

# Adjust layout to prevent overlap
plt.tight_layout()
plt.savefig('D_plot.pdf')
# Show the plot
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import math

# Determine the number of rows
num_rows = lazyResults_Cdf.shape[0]

# Calculate the number of columns and rows needed for subplots
cols = 3  # Number of columns you want in the grid (adjustable)
rows = math.ceil(num_rows / cols)  # Calculate the number of rows needed

# Create a figure to hold all subplots
plt.figure(figsize=(15, rows * 3))  # Adjust the size of the figure

# Loop through all rows and plot each as a histogram
for i, row in enumerate(lazyResults_Cdf.index):
    plt.subplot(rows, cols, i + 1)
    data = lazyResults_Cdf.loc[row]
    
    mean = data.mean()
    std_dev = data.std()
    
    plt.hist(data, bins=50, color='blue', edgecolor='black')
    plt.title(f'Histogram for {row}\nMean = {mean:.2f} ± {std_dev:.2f}')
    plt.xlabel('R-squared Values')
    plt.ylabel('Frequency')

# Adjust layout to prevent overlap
plt.tight_layout()
plt.savefig('C_plot.pdf')
# Show the plot
plt.show()

In [ ]:
import pandas as pd

def addMeanandDev(df):
    df['Mean R-SQD'] = df.mean(axis=1)
    df['Std_Dev'] = df.std(axis=1)
    df = df[['Mean R-SQD', 'Std_Dev'] + [col for col in df.columns if col not in ['Mean R-SQD', 'Std_Dev']]]
    df_sorted = df.sort_values(by='Mean R-SQD', ascending=False)
    mean_row = df_sorted.mean().to_frame().T  # Calculate the mean for each column and transpose it to make it a row
    mean_row.index = ['Column_Mean']  # Give the new row an index name
    df_final = pd.concat([df_sorted, mean_row])
    return df_final

dfQ = pd.read_csv('allQResults_09-06_22-42-46.csv', index_col=0)
dfC = pd.read_csv('allCResults_09-06_22-42-46.csv', index_col=0)
dfD = pd.read_csv('allDResults_09-06_22-42-46.csv', index_col=0)

newDFQ = addMeanandDev(dfQ)
newDFC = addMeanandDev(dfC)
newDFD = addMeanandDev(dfD)
newDFQ.to_csv('new-allQResults_09-06_22-42-46.csv', index=True)
newDFC.to_csv('new-allCResults_09-06_22-42-46.csv', index=True)
newDFD.to_csv('new-allDResults_09-06_22-42-46.csv', index=True)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Define a binary color palette (white for 0, blue for 1)
cmap = sns.color_palette(["white", "blue"])

# Create a heatmap
plt.figure(figsize=(10, 8))  # Adjust size based on your DataFrame
sns.heatmap(sorted_features_df, cmap=cmap, annot=False, cbar=True, linewidths=0.5, linecolor='gray')

# Add title and labels
plt.title('Binary Heatmap of Features Over Runs')
plt.savefig('features.pdf')
plt.show()

In [ ]:
for i in range(50):
    # Perform data splitting using 80/20 ratio

    X_train, X_test, Y_train, Y_test = train_test_split(X_reduced, Y, test_size=0.2, random_state=None)

    docking_features_only = ['Hbond','Hphob','VwInt','Eintl','Dsolv','SolEl','dTSsc']
    XQ_train = X_train.drop(columns=docking_features_only)
    XQ_test = X_test.drop(columns=docking_features_only)

    saveSplitstoFiles(X_train, X_test, Y_train, Y_test, current_time, subdirectory, i+1)

    s = selection_runs[i]
    #s = selected_fs.iloc[:, 0].tolist()

    XD_train = X_train[docking_features_only]
    XD_test = X_test[docking_features_only]
    YD_train = Y_train
    YD_test = Y_test
    clf = LazyRegressor(verbose=0,ignore_warnings=True, custom_metric=None)
    models_train,predictionsD_train = clf.fit(XD_train, XD_train, YD_train, YD_train)
    models_test,predictionsD_test = clf.fit(XD_train, XD_test, YD_train, YD_test)

    predictionsD_test_file = predictionsD_test.reset_index()
    filename = f'predictionsD_test_{current_time}{i}.csv'
    predictionsD_test_file.to_csv(os.path.join(subdirectory, filename), index=False)

    predictionsD_train_aligned = predictionsD_train.reindex(predictionsD_test.index)
    predictionsD_train_file = predictionsD_train_aligned.reset_index()
    filename = f'predictionsD_train_{current_time}{i}.csv'
    predictionsD_train_file.to_csv(os.path.join(subdirectory, filename), index=False)

    XQS_train = XQ_train[s]
    #print(XQS_train)
    XQS_test = XQ_test[s]
    clf = LazyRegressor(verbose=0,ignore_warnings=True, custom_metric=None)
    models_train,predictionsQ_train = clf.fit(XQS_train, XQS_train, Y_train, Y_train)
    models_test,predictionsQ_test = clf.fit(XQS_train, XQS_test, Y_train, Y_test)

    predictionsQ_test_file = predictionsQ_test.reset_index()
    filename = f'predictionsQ_test_{current_time}{i}.csv'
    predictionsQ_test_file.to_csv(os.path.join(subdirectory, filename), index=False)

    predictionsQ_train_aligned = predictionsQ_train.reindex(predictionsQ_test.index)
    predictionsQ_train_file = predictionsQ_train_aligned.reset_index()
    filename = f'predictionsQ_train_{current_time}{i}.csv'
    predictionsQ_train_file.to_csv(os.path.join(subdirectory, filename), index=False)


    docking_features_only_list = list(docking_features_only)
    cmb_features = s + docking_features_only_list
    XC_train = X_train[cmb_features]
    XC_test = X_test[cmb_features]
    clf = LazyRegressor(verbose=0,ignore_warnings=True, custom_metric=None)
    models_train,predictionsC_train = clf.fit(XC_train, XC_train, Y_train, Y_train)
    models_test,predictionsC_test = clf.fit(XC_train, XC_test, Y_train, Y_test)

    predictionsC_test_file = predictionsC_test.reset_index()
    filename = f'predictionsC_test_{current_time}{i}.csv'
    predictionsC_test_file.to_csv(os.path.join(subdirectory, filename), index=False)

    predictionsC_train_aligned = predictionsC_train.reindex(predictionsC_test.index)
    predictionsC_train_file = predictionsC_train_aligned.reset_index()
    filename = f'predictionsC_train_{current_time}{i}.csv'
    predictionsC_train_file.to_csv(os.path.join(subdirectory, filename), index=False)
    
    r_squared_values = predictionsQ_test["R-Squared"]
    
    if i == 0:
        lazyResults_df = pd.DataFrame(index=predictionsQ_test.index)
    lazyResults_df[f'r{i+1}'] = r_squared_values.reindex(lazyResults_df.index)

#print(lazyResults_df)
filename = f'allResults_{current_time}.csv'
lazyResults_df.to_csv(os.path.join(subdirectory, filename), index=True)


In [ ]:
import numpy as np

current_time = datetime.now().strftime('%m-%d_%H-%M-%S')
subdirectory = 'Splits'
all_selected_features = set()
selection_runs = []
lazyResults_df = pd.DataFrame()


for i in range(50):
    # Perform data splitting using 80/20 ratio

    X_train, X_test, Y_train, Y_test = train_test_split(X_reduced, Y, test_size=0.2, random_state=None)

    docking_features_only = ['Hbond','Hphob','VwInt','Eintl','Dsolv','SolEl','dTSsc']
    XQ_train = X_train.drop(columns=docking_features_only)
    XQ_test = X_test.drop(columns=docking_features_only)

    Xf_train, Xf_test, Yf_train, Yf_test = train_test_split(XQ_train, Y_train, test_size=0.2, random_state=42)

    saveSplitstoFiles(X_train, X_test, Y_train, Y_test, current_time, subdirectory, i)

    selected_fs = featureSelectionCoe(Xf_train, Yf_train, current_time, subdirectory, i)
    s = selected_fs.iloc[:, 0].tolist()
    
    selection_runs.append(s)
    all_selected_features.update(s)

    XD_train = X_train[docking_features_only]
    XD_test = X_test[docking_features_only]
    YD_train = Y_train
    YD_test = Y_test
    clf = LazyRegressor(verbose=0,ignore_warnings=True, custom_metric=None)
    models_train,predictionsD_train = clf.fit(XD_train, XD_train, YD_train, YD_train)
    models_test,predictionsD_test = clf.fit(XD_train, XD_test, YD_train, YD_test)

    predictionsD_test_file = predictionsD_test.reset_index()
    filename = f'predictionsD_test_{current_time}{i}.csv'
    predictionsD_test_file.to_csv(os.path.join(subdirectory, filename), index=False)

    predictionsD_train_aligned = predictionsD_train.reindex(predictionsD_test.index)
    predictionsD_train_file = predictionsD_train_aligned.reset_index()
    filename = f'predictionsD_train_{current_time}{i}.csv'
    predictionsD_train_file.to_csv(os.path.join(subdirectory, filename), index=False)

    XQS_train = XQ_train[s]
    #print(XQS_train)
    XQS_test = XQ_test[s]
    clf = LazyRegressor(verbose=0,ignore_warnings=True, custom_metric=None)
    models_train,predictionsQ_train = clf.fit(XQS_train, XQS_train, Y_train, Y_train)
    models_test,predictionsQ_test = clf.fit(XQS_train, XQS_test, Y_train, Y_test)

    predictionsQ_test_file = predictionsQ_test.reset_index()
    filename = f'predictionsQ_test_{current_time}{i}.csv'
    predictionsQ_test_file.to_csv(os.path.join(subdirectory, filename), index=False)

    predictionsQ_train_aligned = predictionsQ_train.reindex(predictionsQ_test.index)
    predictionsQ_train_file = predictionsQ_train_aligned.reset_index()
    filename = f'predictionsQ_train_{current_time}{i}.csv'
    predictionsQ_train_file.to_csv(os.path.join(subdirectory, filename), index=False)


    docking_features_only_list = list(docking_features_only)
    cmb_features = s + docking_features_only_list
    XC_train = X_train[cmb_features]
    XC_test = X_test[cmb_features]
    clf = LazyRegressor(verbose=0,ignore_warnings=True, custom_metric=None)
    models_train,predictionsC_train = clf.fit(XC_train, XC_train, Y_train, Y_train)
    models_test,predictionsC_test = clf.fit(XC_train, XC_test, Y_train, Y_test)

    predictionsC_test_file = predictionsC_test.reset_index()
    filename = f'predictionsC_test_{current_time}{i}.csv'
    predictionsC_test_file.to_csv(os.path.join(subdirectory, filename), index=False)

    predictionsC_train_aligned = predictionsC_train.reindex(predictionsC_test.index)
    predictionsC_train_file = predictionsC_train_aligned.reset_index()
    filename = f'predictionsC_train_{current_time}{i}.csv'
    predictionsC_train_file.to_csv(os.path.join(subdirectory, filename), index=False)
    
    r_squared_values = predictionsQ_test["R-Squared"]
    
    if i == 0:
        lazyResults_df = pd.DataFrame(index=predictionsQ_test.index)
    lazyResults_df[f'r{i+1}'] = r_squared_values.reindex(lazyResults_df.index)

#print(lazyResults_df)
filename = f'allResults_{current_time}.csv'
lazyResults_df.to_csv(os.path.join(subdirectory, filename), index=True)
        
    
all_features_list = list(all_selected_features)
features_df = pd.DataFrame(0, index=range(50), columns=all_features_list)

# Step 4: Populate the DataFrame
for i, selected_features in enumerate(selection_runs):
    features_df.loc[i, selected_features] = 1  # Set selected features to 1 for the i-th run

# View the final DataFrame
#print(features_df)
filename = f'allSfeatures_{current_time}.csv'
features_df.to_csv(os.path.join(subdirectory, filename), index=True)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Define a binary color palette (white for 0, blue for 1)
cmap = sns.color_palette(["white", "blue"])

# Create a heatmap
plt.figure(figsize=(10, 8))  # Adjust size based on your DataFrame
sns.heatmap(features_df, cmap=cmap, annot=False, cbar=True, linewidths=0.5, linecolor='gray')

# Add title and labels
plt.title('Binary Heatmap of Features Over Runs')
plt.savefig('features.pdf')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import math

# Determine the number of rows
num_rows = lazyResults_df.shape[0]

# Calculate the number of columns and rows needed for subplots
cols = 3  # Number of columns you want in the grid (adjustable)
rows = math.ceil(num_rows / cols)  # Calculate the number of rows needed

# Create a figure to hold all subplots
plt.figure(figsize=(15, rows * 3))  # Adjust the size of the figure

# Loop through all rows and plot each as a histogram
for i, row in enumerate(lazyResults_df.index):
    plt.subplot(rows, cols, i + 1)
    plt.hist(lazyResults_df.loc[row], bins=50, color='blue', edgecolor='black')
    plt.title(f'Histogram for {row}')
    plt.xlabel('R-squared Values')
    plt.ylabel('Frequency')

# Adjust layout to prevent overlap
plt.tight_layout()
plt.savefig('my_plot1.pdf')
# Show the plot
plt.show()

In [ ]:
first_five_rows = lazyResults_df.head(3)

plt.figure(figsize=(12, 10))

# Loop through the first 5 rows and plot each as a histogram
for i, row in enumerate(first_five_rows.index):
    plt.subplot(3, 2, i + 1)  # 3 rows and 2 columns for the grid of plots
    plt.hist(lazyResults_df.loc[row], bins=10, color='blue', edgecolor='black')
    plt.title(f'Histogram for {row}')
    plt.xlabel('R-squared Values')
    plt.ylabel('Frequency')

# Adjust layout and show the plot
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# Perform data splitting using 50/50 ratio
# do loop for random state to check which one is best
#subject accuracy`
#morgan fingerpriting
X_train, X_test, Y_train, Y_test = train_test_split(X_reduced, Y, test_size=0.2, random_state=None)
print(X_train)

In [ ]:
docking_features_only = ['Hbond','Hphob','VwInt','Eintl','Dsolv','SolEl','dTSsc']
XQ_train = X_train.drop(columns=docking_features_only)
XQ_test = X_test.drop(columns=docking_features_only)

In [ ]:
Xf_train, Xf_test, Yf_train, Yf_test = train_test_split(XQ_train, Y_train, test_size=0.2, random_state=42)

Using Ridge regress + cross validation on Training data to apply feature selections

In [ ]:
import os
from datetime import datetime

##top_selection = X_ext[top_predictors]
#top_selection = X[selected_predictors] #this is for descriptors only since not extended file so no Hbond etc.
#top_selection.shape

# Perform data splitting using 50/50 ratio
##X1_train, X1_test, Y1_train, Y1_test = train_test_split(top_selection, Y, test_size=0.5, random_state=42)

# Save Split to files
# Specify the subdirectory
subdirectory = 'Splits'

# Get the current time and format it as a string
current_time = datetime.now().strftime('%m-%d_%H-%M-%S')

# Combine the subdirectory and file name into a full path
#full_path = os.path.join(subdirectory, filename)

# Create the subdirectory if it doesn't exist
os.makedirs(subdirectory, exist_ok=True)
# Create the file name with the current time
filename = f'XTrain_{current_time}.csv'
X_train.to_csv(os.path.join(subdirectory, filename), index=True)
filename = f'XTest_{current_time}.csv'
X_test.to_csv(os.path.join(subdirectory, filename), index=True)
filename = f'YTrain_{current_time}.csv'
Y_train.to_csv(os.path.join(subdirectory, filename), index=True)
filename = f'YTest_{current_time}.csv'
Y_test.to_csv(os.path.join(subdirectory, filename), index=True)

# Defines and builds the lazyclassifier
#clf = LazyRegressor(verbose=0,ignore_warnings=True, custom_metric=None)
#models_train,predictions_train = clf.fit(X1_train, X1_train, Y1_train, Y1_train)
#models_test,predictions_test = clf.fit(X1_train, X1_test, Y1_train, Y1_test)

In [ ]:
# Good method Ridge regression + cross validation
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeCV, Ridge
from sklearn.model_selection import cross_val_score

# Step 1: Standardize the features

scaler = StandardScaler()
X_scaled = scaler.fit_transform(Xf_train)

print("Are there NaN values in X_scaled?:", np.isnan(X_scaled).any())
print("Number of NaN values in each column:\n", np.isnan(X_scaled).sum(axis=0))
X_scaled_clean = X_scaled[~np.isnan(X_scaled).any(axis=1)]
Y_clean = Yf_train[~np.isnan(X_scaled).any(axis=1)]

# Step 2: Use RidgeCV to automatically find the best alpha
alphas = np.logspace(-6, 6, 13)  # Range of alpha values to test
#alphas = np.logspace(np.log10(best_alpha) - 2, np.log10(best_alpha) + 2, 15)
ridge_cv = RidgeCV(alphas=alphas, cv=5, scoring='neg_mean_squared_error')
ridge_cv.fit(X_scaled_clean, Y_clean)

# Step 3: Fit the Ridge model with the optimal alpha
best_alpha = ridge_cv.alpha_
ridge = Ridge(alpha=best_alpha, max_iter=10000, random_state=42)
ridge.fit(X_scaled_clean, Y_clean)

# Step 3: Get the absolute values of the coefficients
coef_abs = np.abs(ridge.coef_)

# Step 4: Rank features based on the magnitude of their coefficients
coef_ranking = pd.Series(coef_abs, index=Xf_train.columns).sort_values(ascending=False)

# Step 5: Select top N features
N = 12  # Example: Select the top 50 features
selected_features_ridge = coef_ranking.head(N).index
print(selected_features_ridge)
#cmb_selected_features = selected_features.union(additional_predictors)
# Convert selected features to a DataFrame
df_selected_features = pd.DataFrame(selected_features_ridge, columns=['Selected Features'])

# Save to CSV
filename = f'selected_features_{current_time}.csv'
df_selected_features.to_csv(os.path.join(subdirectory, filename), index=False)

In [ ]:
# Defines and builds the lazyclassifier
# Run on Docking features
XD_train = X_train[docking_features_only]
XD_test = X_test[docking_features_only]
YD_train = Y_train
YD_test = Y_test
clf = LazyRegressor(verbose=0,ignore_warnings=True, custom_metric=None)
models_train,predictionsD_train = clf.fit(XD_train, XD_train, YD_train, YD_train)
models_test,predictionsD_test = clf.fit(XD_train, XD_test, YD_train, YD_test)
# Sort models by R-squared in descending order and get the top 3 models
top_3_models = models_test.sort_values(by="R-Squared", ascending=False).head(3)

# Print the top 3 models and their parameters
for model_name in top_3_models.index:
    model = clf.models[model_name]
    print(f"Model: {model_name}")
    print(f"Parameters: {model.get_params()}")
    print("-" * 40)

## Additional experimental branches

These cells explore alternatives and depend on the selected analysis branch and its variables.


In [ ]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import GridSearchCV, cross_val_score, KFold
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoLarsIC
# Create a pipeline that combines feature selection and the model
pipeline = Pipeline([
    ('feature_selection', SelectKBest(score_func=f_regression, k=10)),  # Select top 25 features
    ('regressor', RandomForestRegressor(random_state=42))
])

# Outer cross-validation (4-fold)
outer_cv = KFold(n_splits=4, shuffle=True, random_state=42)

# Inner cross-validation (3-fold for hyperparameter tuning)
inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

# Define the parameter grid for the inner loop (for the model only, as we fix k=25 for features)
param_grid = {
    'regressor__n_estimators': [100, 200, 500],  # Number of trees in the forest
    'regressor__max_depth': [None, 10, 20, 30],  # Maximum depth of the tree
    'regressor__min_samples_split': [2, 5, 10],  # Minimum number of samples required to split a node
    'regressor__min_samples_leaf': [1, 2, 4],  # Minimum number of samples required at each leaf node
    'regressor__bootstrap': [True, False]  # Whether bootstrap samples are used when building trees
}

# GridSearchCV for hyperparameter tuning within the inner loop
clf = GridSearchCV(pipeline, param_grid, cv=inner_cv, scoring='neg_mean_squared_error', n_jobs=-1)

# Perform 4-fold nested cross-validation
nested_scores = cross_val_score(clf, Xf_train, Yf_train, cv=outer_cv, scoring='neg_mean_squared_error')

# Print the results
print("Nested CV MSE scores: ", -nested_scores)
print("Mean MSE: ", -np.mean(nested_scores))

# Fit the model with the best hyperparameters on the entire training set
clf.fit(Xf_train, Yf_train)

# Get the best model and print its hyperparameters
best_model = clf.best_estimator_
print("-----Best Model-----")
print(best_model.named_steps['regressor'])

# Get the SelectKBest object from the pipeline
selector = clf.best_estimator_.named_steps['feature_selection']

# Get the mask of selected features (True for selected, False for not)
selected_features_mask = selector.get_support()

# Get the indices of the selected features
selected_features_indices = np.where(selected_features_mask)[0]

# If you have feature names, you can map the indices to names
selected_features_cmb = feature_names[selected_features_mask]

# Print the selected features
print("Top 25 selected features (indices):", selected_features_indices)
print("Top 25 selected features (names):", selected_features_cmb)

In [ ]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import GridSearchCV, cross_val_score, KFold
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.linear_model import LassoLarsIC

# Create a pipeline that combines feature selection and the model
pipeline = Pipeline([
    ('feature_selection', SelectKBest(score_func=f_regression, k=10)),  # Select top 10 features
    ('regressor', SVR())
])

# Outer cross-validation (4-fold)
outer_cv = KFold(n_splits=4, shuffle=True, random_state=42)

# Inner cross-validation (3-fold for hyperparameter tuning)
inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

# Define the parameter grid for the inner loop (for the model only, as we fix k=10 for features)
param_grid = {
    'regressor__C': [0.1, 1, 10],  # SVR regularization parameter
    'regressor__kernel': ['linear', 'rbf']  # SVR kernel type
}

# GridSearchCV for hyperparameter tuning within the inner loop
clf = GridSearchCV(pipeline, param_grid, cv=inner_cv, scoring='neg_mean_squared_error', n_jobs=-1)

# Perform 4-fold nested cross-validation
nested_scores = cross_val_score(clf, XQ_train, Y_train, cv=outer_cv, scoring='neg_mean_squared_error')

# Print the results
print("Nested CV MSE scores: ", -nested_scores)
print("Mean MSE: ", -np.mean(nested_scores))

feature_names = XQ_train.columns

clf.fit(XQ_train, Y_train)

best_model = clf.best_estimator_
print("-----Best Model-----")
print(best_model.named_steps['regressor'])

# Get the SelectKBest object from the pipeline
selector = clf.best_estimator_.named_steps['feature_selection']

# Get the mask of selected features (True for selected, False for not)
selected_features_mask = selector.get_support()

# Get the indices of the selected features
selected_features_indices = np.where(selected_features_mask)[0]

# If you have feature names, you can map the indices to names
# Assuming you have feature names in a list or array:
# feature_names = ['feature1', 'feature2', ..., 'feature20'] (replace with actual names)
# For example:
selected_features_cmb = feature_names[selected_features_mask]

# Print the selected features
print("Top 10 selected features (indices):", selected_features_indices)
print("Top 10 selected features (names):", selected_features_cmb)

In [ ]:
import shap

# Select the reduced dataset with only the top 10 features
X_train_selected = Xf_train.iloc[:, selected_features_indices]
X_test_selected = Xf_test.iloc[:, selected_features_indices]

# Use the fitted SVR model (best_model.named_steps['regressor']) to predict on the reduced feature set
explainer = shap.KernelExplainer(best_model.named_steps['regressor'].predict, X_train_selected)

# Compute SHAP values for the test set
shap_values = explainer.shap_values(X_test_selected)

# Plot the SHAP summary plot for the selected features
shap.summary_plot(shap_values, X_test_selected, feature_names=selected_features_cmb)

In [ ]:
pip install shap

In [ ]:
import shap

# Select the reduced dataset with only the top 10 features
X_train_selected = Xf_train.iloc[:, selected_features_indices]
X_test_selected = Xf_test.iloc[:, selected_features_indices]

# Use the fitted SVR model (best_model.named_steps['regressor']) to predict on the reduced feature set
explainer = shap.KernelExplainer(best_model.named_steps['regressor'].predict, X_train_selected)

# Compute SHAP values for the test set
shap_values = explainer.shap_values(X_test_selected)

# Plot the SHAP summary plot for the selected features
shap.summary_plot(shap_values, X_test_selected, feature_names=selected_features_cmb)

In [ ]:
pip install mlxtend

In [ ]:
#Use SelectKBest to reduce features to 30
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import train_test_split

# Select the top 50 features using SelectKBest
selector = SelectKBest(score_func=f_regression, k=50)
X_train_selected = selector.fit_transform(Xf_train, Yf_train)
X_test_selected = selector.transform(Xf_test)

# Get the selected feature indices or names
selected_feature_indices = selector.get_support(indices=True)
selected_feature_names = X.columns[selected_feature_indices]  # If X is a DataFrame

print("Selected feature indices:", selected_feature_indices)

In [ ]:
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.neighbors import KNeighborsRegressor

models = {
    'SVR': SVR(),
    'RandomForest': RandomForestRegressor(),
    'Lasso': Lasso(),
    'Ridge': Ridge(),
    'ElasticNet': ElasticNet(),
    'XGBoost': XGBRegressor(),
    #'LightGBM': LGBMRegressor(),
    'GradientBoosting': GradientBoostingRegressor(),
    'AdaBoost': AdaBoostRegressor(),
    'ExtraTrees': ExtraTreesRegressor(),
    'KNN': KNeighborsRegressor()
}

feature_sets = {}
print("starting....")
for model_name, model in models.items():
    sfs = SFS(model, 
              k_features='best', 
              forward=True, 
              floating=False, 
              scoring='r2', 
              cv=5)
    
    sfs = sfs.fit(X_train_selected, Yf_train)
    selected_feature_indices = sfs.k_feature_idx_
    selected_feature_names = [feature_names[i] for i in selected_feature_indices]
    
    feature_sets[model_name] = sfs.k_feature_idx_
    
    print(f"Model: {model_name}")
    print(f"Selected Features: {list(selected_feature_names)}\n")

print(feature_sets)

In [ ]:
import shap

explainer = shap.LinearExplainer(ridge, XQ_train)
shap_values = explainer.shap_values(XQ_test)
shap.summary_plot(shap_values, XQ_test)

explainer2 = shap.LinearExplainer(best_model.named_steps['regressor'], XQ_train)
shap_values2 = explainer2.shap_values(XQ_test)
shap.summary_plot(shap_values2, XQ_test)

Filter predictors more with coefficency

In [ ]:
#heatmap for coefficiency selection
linear_models = {
    'LarsCV': LarsCV(),
    'SGDRegressor': SGDRegressor(max_iter=1000, tol=1e-3),
    'OrthogonalMatchingPursuit': OrthogonalMatchingPursuit(),
    'OrthogonalMatchingPursuitCV': OrthogonalMatchingPursuitCV(),
    #'NuSVR': NuSVR(),
    'RidgeCV': RidgeCV(),
    #'SVR': SVR(),
    'PoissonRegressor': PoissonRegressor(),
    'GammaRegressor': GammaRegressor(),
    'BayesianRidge': BayesianRidge(),
    'LassoCV': LassoCV(),
    'ElasticNetCV': ElasticNetCV(),
    'LassoLarsCV': LassoLarsCV()
}

tree_based_models = {
    'KNeighborsRegressor': KNeighborsRegressor(),
    'AdaBoostRegressor': AdaBoostRegressor(),
    'LGBMRegressor': LGBMRegressor(),
    'HistGradientBoostingRegressor': HistGradientBoostingRegressor(),
    'BaggingRegressor': BaggingRegressor(),
    'GradientBoostingRegressor': GradientBoostingRegressor(),
    'TweedieRegressor': TweedieRegressor(),
    'XGBRegressor': XGBRegressor(),
    'RandomForestRegressor': RandomForestRegressor()
}

# Create empty DataFrames to store coefficients and feature importances
linear_coeffs_df = pd.DataFrame(columns=X_train.columns)
tree_importances_df = pd.DataFrame(columns=X_train.columns)

# Fit each linear model and collect coefficients
for name, model in linear_models.items():
    model.fit(X_train, Y_train)
    if hasattr(model, 'coef_'):
        coefficients = model.coef_
    else:
        coefficients = np.full(X_train.shape[1], np.nan)
    
    linear_coeffs_df.loc[name] = coefficients
    linear_coeffs_df = linear_coeffs_df.clip(upper=1)

# Fit each tree-based model and collect feature importances
for name, model in tree_based_models.items():
    model.fit(X_train, Y_train)
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
    else:
        importances = np.full(X_train.shape[1], np.nan)
    
    tree_importances_df.loc[name] = importances


# Normalize the feature importances to be on a similar scale as coefficients
tree_importances_df = tree_importances_df.div(tree_importances_df.max(axis=1), axis=0)

# Generate a heatmap for the linear model coefficients
plt.figure(figsize=(15, 10))
sns.heatmap(linear_coeffs_df, cmap='coolwarm', annot=False)
plt.title('Heatmap of Linear Model Coefficients')
plt.xlabel('Features')
plt.ylabel('Models')
plt.savefig('heatmap_feature_selection.png', dpi=300, bbox_inches='tight')
plt.show()

# Generate a heatmap for the tree-based model feature importances
plt.figure(figsize=(15, 10))
sns.heatmap(tree_importances_df, cmap='coolwarm', annot=False)
plt.title('Heatmap of Tree-Based Model Feature Selection')
plt.xlabel('Features')
plt.ylabel('Models')
plt.show()

Compare models Save slice

In [ ]:
# Defines and builds the lazyclassifier
# Run on Docking features
XD_train = X_train[docking_features_only]
XD_test = X_test[docking_features_only]
YD_train = Y_train
YD_test = Y_test
clf = LazyRegressor(verbose=0,ignore_warnings=True, custom_metric=None)
models_train,predictionsD_train = clf.fit(XD_train, XD_train, YD_train, YD_train)
models_test,predictionsD_test = clf.fit(XD_train, XD_test, YD_train, YD_test)
# Sort models by R-squared in descending order and get the top 3 models
top_3_models = models_test.sort_values(by="R-Squared", ascending=False).head(3)

# Print the top 3 models and their parameters
for model_name in top_3_models.index:
    model = clf.models[model_name]
    print(f"Model: {model_name}")
    print(f"Parameters: {model.get_params()}")
    print("-" * 40)

In [ ]:
predictionsD_test

In [ ]:
predictionsD_test_file = predictionsD_test.reset_index()
filename = f'predictionsD_test_{current_time}.csv'
predictionsD_test_file.to_csv(os.path.join(subdirectory, filename), index=False)

In [ ]:
predictionsD_train_aligned = predictionsD_train.reindex(predictionsD_test.index)
predictionsD_train_aligned

In [ ]:
predictionsD_train_file = predictionsD_train_aligned.reset_index()
filename = f'predictionsD_train_{current_time}.csv'
predictionsD_train_file.to_csv(os.path.join(subdirectory, filename), index=False)

In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.svm import NuSVR

# Define the numeric and categorical features based on your provided columns
numeric_features = ['Hbond', 'Hphob', 'VwInt', 'Eintl', 'Dsolv', 'SolEl', 'dTSsc']

categorical_low_features = []  # Add any low-cardinality categorical features here
categorical_high_features = []  # Add any high-cardinality categorical features here

# Define the preprocessor for numerical and categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', Pipeline(steps=[
            ('imputer', SimpleImputer()),  # Handle missing values
            ('scaler', StandardScaler())  # Standardize the numeric data
        ]), numeric_features),
        
        ('categorical_low', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('encoding', OneHotEncoder(handle_unknown='ignore', sparse=False))
        ]), categorical_low_features),
        
        ('categorical_high', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('encoding', OrdinalEncoder())
        ]), categorical_high_features)
    ]
)

# Define the final pipeline including the preprocessor and the regressor
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', NuSVR(
        C=1.0,
        kernel='rbf',
        degree=3,
        gamma='scale',
        coef0=0.0,
        shrinking=True,
        tol=0.001,
        cache_size=200,
        verbose=False,
        max_iter=-1,
        nu=0.5
    ))
])

# Fit the model
pipeline.fit(XD_train, Y_train)

# Make predictions
Y_pred = pipeline.predict(XD_test)

# Calculate Mean Squared Error (optional)
mse = mean_squared_error(YD_test, Y_pred)
print(f"Mean Squared Error: {mse}")
# Calculate R-squared value
r_squared = r2_score(Y_test, Y_pred)
print(f"R-squared: {r_squared}")

# Plotting Actual vs Predicted
plt.figure(figsize=(10, 10))
plt.scatter(Y_test, Y_pred, color='blue', label='Predicted vs Actual')
plt.plot([min(Y_test), max(Y_test)], [min(Y_test), max(Y_test)], color='red', lw=2, label='Ideal Fit')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title(f'Actual vs Predicted using NuSVR')
# Add R-squared value to the plot
plt.text(0.05, 0.95, f'R-squared = {r_squared:.2f}', transform=plt.gca().transAxes, fontsize=12, verticalalignment='top')

plt.grid(True)
#plt.legend()
plt.savefig("dockingplot.png")
plt.show()

In [ ]:
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer

# Define your feature sets
numeric_features = ['Hbond', 'Hphob', 'VwInt', 'Eintl', 'Dsolv', 'SolEl', 'dTSsc']
categorical_low_features = []  # Replace with actual low-cardinality categorical features
categorical_high_features = []  # Replace with actual high-cardinality categorical features

# Preprocessor setup
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler())]), numeric_features),
        ('categorical_low', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('encoding', OneHotEncoder(handle_unknown='ignore', sparse=False))]), categorical_low_features),
        ('categorical_high', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('encoding', OrdinalEncoder())]), categorical_high_features)
    ]
)

# SVR model
svr = SVR(C=1.0, kernel='rbf', gamma='scale', epsilon=0.1)

# Full pipeline
model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('regressor', svr)])

# Fit the model
model_pipeline.fit(XD_train, Y_train)

# Make predictions
Y_pred = model_pipeline.predict(XD_test)

# Calculate Mean Squared Error (optional)
mse = mean_squared_error(YD_test, Y_pred)
print(f"Mean Squared Error: {mse}")
# Calculate R-squared value
r_squared = r2_score(Y_test, Y_pred)
print(f"R-squared: {r_squared}")

# Plotting Actual vs Predicted
plt.figure(figsize=(10, 10))
plt.scatter(Y_test, Y_pred, color='blue', label='Predicted vs Actual')
plt.plot([min(Y_test), max(Y_test)], [min(Y_test), max(Y_test)], color='red', lw=2, label='Ideal Fit')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title(f'Actual vs Predicted using SVR: (R² : {r_squared:.2f})')
plt.grid(True)
#plt.legend()
plt.show()

In [ ]:
# PUBLIC COPY: hard-coded private feature identities are redacted placeholders.
#best s2=['REDACTED_FEATURE_001', 'REDACTED_FEATURE_002', 'REDACTED_FEATURE_003', 'REDACTED_FEATURE_004', 'REDACTED_FEATURE_005', 'REDACTED_FEATURE_006', 'REDACTED_FEATURE_007', 'REDACTED_FEATURE_008', 'REDACTED_FEATURE_009', 'REDACTED_FEATURE_010', 'REDACTED_FEATURE_011', 'REDACTED_FEATURE_012']
#second best s2 = ['REDACTED_FEATURE_001', 'REDACTED_FEATURE_002', 'REDACTED_FEATURE_003', 'REDACTED_FEATURE_013', 'REDACTED_FEATURE_014', 'REDACTED_FEATURE_015', 'REDACTED_FEATURE_016', 'REDACTED_FEATURE_005', 'REDACTED_FEATURE_006', 'REDACTED_FEATURE_007', 'REDACTED_FEATURE_009', 'REDACTED_FEATURE_010', 'REDACTED_FEATURE_017', 'REDACTED_FEATURE_018', 'REDACTED_FEATURE_011', 'REDACTED_FEATURE_019', 'REDACTED_FEATURE_012']


# Defines and builds the lazyclassifier
# Run on QSAR features
s = selected_features_ridge
s1 = ['REDACTED_FEATURE_001', 'REDACTED_FEATURE_002', 'REDACTED_FEATURE_005', 'REDACTED_FEATURE_020', 'REDACTED_FEATURE_021']
#s2 = ['REDACTED_FEATURE_001', 'REDACTED_FEATURE_002', 'REDACTED_FEATURE_013', 'REDACTED_FEATURE_005', 'REDACTED_FEATURE_014']
#s2 = ['REDACTED_FEATURE_001', 'REDACTED_FEATURE_002', 'REDACTED_FEATURE_003', 'REDACTED_FEATURE_013', 'REDACTED_FEATURE_014', 'REDACTED_FEATURE_015']
#s2_best = ['REDACTED_FEATURE_001', 'REDACTED_FEATURE_002', 'REDACTED_FEATURE_003', 'REDACTED_FEATURE_013', 'REDACTED_FEATURE_014', 'REDACTED_FEATURE_015', 'REDACTED_FEATURE_016', 'REDACTED_FEATURE_005', 'REDACTED_FEATURE_006', 'REDACTED_FEATURE_007', 'REDACTED_FEATURE_009', 'REDACTED_FEATURE_010', 'REDACTED_FEATURE_017', 'REDACTED_FEATURE_018', 'REDACTED_FEATURE_011', 'REDACTED_FEATURE_019', 'REDACTED_FEATURE_012']
s2 = ['REDACTED_FEATURE_001', 'REDACTED_FEATURE_004', 'REDACTED_FEATURE_015', 'REDACTED_FEATURE_005', 'REDACTED_FEATURE_006', 'REDACTED_FEATURE_007', 'REDACTED_FEATURE_009', 'REDACTED_FEATURE_010', 'REDACTED_FEATURE_011', 'REDACTED_FEATURE_019', 'REDACTED_FEATURE_012']
#s2 = ['REDACTED_FEATURE_001', 'REDACTED_FEATURE_002', 'REDACTED_FEATURE_004', 'REDACTED_FEATURE_003', 'REDACTED_FEATURE_013', 'REDACTED_FEATURE_014', 'REDACTED_FEATURE_022', 'REDACTED_FEATURE_023', 'REDACTED_FEATURE_024', 'REDACTED_FEATURE_016', 'REDACTED_FEATURE_025', 'REDACTED_FEATURE_026', 'REDACTED_FEATURE_027', 'REDACTED_FEATURE_028']
#s2=['REDACTED_FEATURE_015', 'REDACTED_FEATURE_023', 'REDACTED_FEATURE_029', 'REDACTED_FEATURE_030', 'REDACTED_FEATURE_005', 'REDACTED_FEATURE_031', 'REDACTED_FEATURE_028', 'REDACTED_FEATURE_032']
#s2=['REDACTED_FEATURE_001', 'REDACTED_FEATURE_002', 'REDACTED_FEATURE_004', 'REDACTED_FEATURE_003', 'REDACTED_FEATURE_014', 'REDACTED_FEATURE_022', 'REDACTED_FEATURE_023', 'REDACTED_FEATURE_033', 'REDACTED_FEATURE_034', 'REDACTED_FEATURE_035', 'REDACTED_FEATURE_036', 'REDACTED_FEATURE_031', 'REDACTED_FEATURE_037', 'REDACTED_FEATURE_028', 'REDACTED_FEATURE_008', 'REDACTED_FEATURE_038', 'REDACTED_FEATURE_039', 'REDACTED_FEATURE_040', 'REDACTED_FEATURE_041']
#s2=['REDACTED_FEATURE_001', 'REDACTED_FEATURE_002', 'REDACTED_FEATURE_004', 'REDACTED_FEATURE_003', 'REDACTED_FEATURE_014', 'REDACTED_FEATURE_022', 'REDACTED_FEATURE_023', 'REDACTED_FEATURE_033', 'REDACTED_FEATURE_034', 'REDACTED_FEATURE_035', 'REDACTED_FEATURE_036', 'REDACTED_FEATURE_031', 'REDACTED_FEATURE_037', 'REDACTED_FEATURE_028', 'REDACTED_FEATURE_008', 'REDACTED_FEATURE_038', 'REDACTED_FEATURE_039', 'REDACTED_FEATURE_040', 'REDACTED_FEATURE_041']
###s2=['REDACTED_FEATURE_001', 'REDACTED_FEATURE_002', 'REDACTED_FEATURE_003', 'REDACTED_FEATURE_004', 'REDACTED_FEATURE_005', 'REDACTED_FEATURE_006', 'REDACTED_FEATURE_007', 'REDACTED_FEATURE_008', 'REDACTED_FEATURE_009', 'REDACTED_FEATURE_010', 'REDACTED_FEATURE_011', 'REDACTED_FEATURE_012']
#s2=['REDACTED_FEATURE_001', 'REDACTED_FEATURE_002', 'REDACTED_FEATURE_004', 'REDACTED_FEATURE_014', 'REDACTED_FEATURE_022', 'REDACTED_FEATURE_023', 'REDACTED_FEATURE_042', 'REDACTED_FEATURE_043', 'REDACTED_FEATURE_035', 'REDACTED_FEATURE_005', 'REDACTED_FEATURE_037', 'REDACTED_FEATURE_044', 'REDACTED_FEATURE_008', 'REDACTED_FEATURE_045', 'REDACTED_FEATURE_007', 'REDACTED_FEATURE_041']
XQS_train = XQ_train[s]
XQS_test = XQ_test[s]
clf = LazyRegressor(verbose=0,ignore_warnings=True, custom_metric=None)
models_train,predictionsQ_train = clf.fit(XQS_train, XQS_train, Y_train, Y_train)
models_test,predictionsQ_test = clf.fit(XQS_train, XQS_test, Y_train, Y_test)
# Sort models by R-squared in descending order and get the top 3 models
top_3_models = models_test.sort_values(by="R-Squared", ascending=False).head(3)
# Print the top 3 models and their parameters
for model_name in top_3_models.index:
    model = clf.models[model_name]
    print(f"Model: {model_name}")
    print(f"Parameters: {model.get_params()}")
    print("-" * 40)

In [ ]:
import pandas as pd
all_top_features_list = pd.read_csv('Splits/Run2/allTopFeatures_09-06_22-42-46.csv')
top_features_df = pd.DataFrame(0, index=range(50), columns=all_top_features_list)
# Step 4: Populate the DataFrame
for i, selected_features in enumerate(top_selection_runs):
    top_features_df.loc[i, selected_features] = 1  # Set selected features to 1 for the i-th run

# View the final DataFrame
#print(features_df)
#filename = f'allSfeatures_{current_time}.csv'
#features_df.to_csv(os.path.join(subdirectory, filename), index=True)

top_feature_sums = top_features_df.sum(axis=0)
top_sorted_features = top_feature_sums.sort_values(ascending=False)

top_sorted_features_df = top_features_df[top_sorted_features.index]
print(top_sorted_features_df)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Define a colormap
cmap = sns.color_palette("coolwarm", as_cmap=True)
plt.figure(figsize=(12, 12))  # Adjust figure size
#sns.clustermap(sorted_features_df, cmap=sns.color_palette(["black", "#00FF00"], as_cmap=True), linewidths=0, yticklabels=False, col_cluster=False, row_cluster=True)
sns.clustermap(
    top_sorted_features_df,
    cmap=cmap,
    annot=False,
    cbar=True,
    linewidths=0.5,
    linecolor='gray',
    figsize=(10, 8),  # Adjust the size of the heatmap
    method='average',  # Clustering method (e.g., 'single', 'complete', 'average', etc.)
    metric='euclidean',  # Distance metric (e.g., 'euclidean', 'manhattan', etc.)
    row_cluster=True,  # Perform clustering on rows
    col_cluster=False  # Do not perform clustering on columns (only on rows)
)
plt.savefig('features.pdf')
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
import lazypredict
from lazypredict.Supervised import LazyRegressor

In [ ]:
# PUBLIC COPY: hard-coded private feature identities are redacted placeholders.
#Run directly from saved files

#df_all_features = pd.read_csv('Splits/Run2/allTopFeatures_09-06_22-42-46.csv', header=None)
#f_list = df_all_features.values.tolist()
s_features = ['REDACTED_FEATURE_001','REDACTED_FEATURE_002','REDACTED_FEATURE_021','REDACTED_FEATURE_004','REDACTED_FEATURE_005','REDACTED_FEATURE_020','REDACTED_FEATURE_046','REDACTED_FEATURE_011','REDACTED_FEATURE_012','REDACTED_FEATURE_014','REDACTED_FEATURE_015','REDACTED_FEATURE_047']

clf = LazyRegressor(verbose=0,ignore_warnings=True,custom_metric=None)
#count =0
#for s_features in f_list:

X17_train = pd.read_csv('Splits/Run2/_XTrain_09-06_22-42-46_17.csv')
X17_train = X17_train.iloc[:, 1:]
X17_test = pd.read_csv('Splits/Run2/_XTest_09-06_22-42-46_17.csv')
X17_test = X17_test.iloc[:, 1:]
Y17_train = pd.read_csv('Splits/Run2/_YTrain_09-06_22-42-46_17.csv')
Y17_train = Y17_train.iloc[:, 1:]
Y17_test = pd.read_csv('Splits/Run2/_YTest_09-06_22-42-46_17.csv')
Y17_test = Y17_test.iloc[:, 1:]

XQS_train = X17_train[s_features]

XQS_test = X17_test[s_features]
models_train,predictionsQ_train = clf.fit(XQS_train, XQS_train, Y17_train, Y17_train)
models_test,predictionsQ_test = clf.fit(XQS_train, XQS_test, Y17_train, Y17_test)
print(predictionsQ_test)
        
# Sort models by R-squared in descending order and get the top 3 models
top_3_models = models_test.sort_values(by="R-Squared", ascending=False).head()
# Print the top 3 models and their parameters
for model_name in top_3_models.index:
    if model_name == "NuSVR":
        model = clf.models[model_name]
        print(f"Model: {model_name}")
        print(f"Parameters: {model.get_params()}")
        print("-" * 40)
        break

In [ ]:
# PUBLIC COPY: hard-coded private feature identities are redacted placeholders.
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.svm import NuSVR
import pandas as pd

# Define the feature sets
numeric_features = ['REDACTED_FEATURE_001', 'REDACTED_FEATURE_002', 'REDACTED_FEATURE_021', 'REDACTED_FEATURE_004',
                    'REDACTED_FEATURE_005', 'REDACTED_FEATURE_020', 'REDACTED_FEATURE_046', 'REDACTED_FEATURE_011',
                    'REDACTED_FEATURE_012', 'REDACTED_FEATURE_014', 'REDACTED_FEATURE_015', 'REDACTED_FEATURE_047']
categorical_low_features = []  # No categorical_low features in this example
categorical_high_features = []  # No categorical_high features in this example

# Preprocessing pipeline for numeric features
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),  # Impute missing values using the mean
    ('scaler', StandardScaler())  # Standardize the numeric features
])

# Preprocessing pipeline for categorical (low cardinality) features
categorical_low_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),  # Impute missing values with 'missing'
    ('encoding', OneHotEncoder(handle_unknown='ignore', sparse_output=False))  # One-hot encoding
])

# Preprocessing pipeline for categorical (high cardinality) features
categorical_high_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),  # Impute missing values with 'missing'
    ('encoding', OrdinalEncoder())  # Ordinal encoding
])

# Combine all preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_transformer, numeric_features),
        ('categorical_low', categorical_low_transformer, categorical_low_features),
        ('categorical_high', categorical_high_transformer, categorical_high_features)
    ]
)

# Define the NuSVR model with the provided parameters
nusvr_regressor = NuSVR(
    C=1.0,               # Regularization parameter
    nu=0.5,              # Nu parameter
    kernel='rbf',         # Kernel type: Radial Basis Function (RBF)
    gamma='scale',        # Kernel coefficient
    tol=0.001,            # Tolerance for stopping criterion
    shrinking=True,       # Use the shrinking heuristic
    cache_size=200,       # Cache size for kernel computation
    verbose=False         # Suppress verbosity
)

# Create the final pipeline with preprocessing and NuSVR regressor
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', nusvr_regressor)
])

# Fit the model
model.fit(XQS_train, Y17_train)
Y_test_values = Y17_test['pIC50'].values

# Predict using the test set
Y_pred = model.predict(XQS_test)

# Evaluate and plot actual vs predicted
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score

# Plot the Actual vs Predicted values
plt.figure(figsize=(10, 10))
plt.scatter(Y_test_values, Y_pred, color='blue', label='Predicted vs Actual')

# Perfect fit line
plt.plot([min(Y_test_values), max(Y_test_values)], [min(Y_test_values), max(Y_test_values)], color='red', label='Perfect Fit')

plt.title('QSAR Features: Actual vs Predicted using NuSVR', fontsize=20)
plt.xlabel('Actual pIC50 Values', fontsize=20)
plt.ylabel('Predicted pIC50 Values', fontsize=20)
plt.legend(fontsize=20)
# Add R-squared value to the plot
plt.text(0.04, 0.84, f'R-squared = {r2_score(Y_test_values, Y_pred):.2f}', transform=plt.gca().transAxes, fontsize=20, verticalalignment='top')

plt.grid(True)
#plt.legend()
plt.savefig("qsar-plot-big.pdf")
plt.grid(True)
plt.show()

# Print performance metrics
print(f"Mean Squared Error: {mean_squared_error(Y_test_values, Y_pred)}")
print(f"R-Squared: {r2_score(Y_test_values, Y_pred)}")

In [ ]:
# PUBLIC COPY: hard-coded private feature identities are redacted placeholders.
#Run directly from saved files

#df_all_features = pd.read_csv('Splits/Run2/allTopFeatures_09-06_22-42-46.csv', header=None)
#f_list = df_all_features.values.tolist()
s_features = ['REDACTED_FEATURE_001','REDACTED_FEATURE_002','REDACTED_FEATURE_021','REDACTED_FEATURE_004','REDACTED_FEATURE_005','REDACTED_FEATURE_020','REDACTED_FEATURE_046','REDACTED_FEATURE_011','REDACTED_FEATURE_012','REDACTED_FEATURE_014','REDACTED_FEATURE_015','REDACTED_FEATURE_047']
docking_features_only = ['Hbond','Hphob','VwInt','Eintl','Dsolv','SolEl','dTSsc']

clf = LazyRegressor(verbose=0,ignore_warnings=True,custom_metric=None)
#count =0
#for s_features in f_list:

X17_train = pd.read_csv('Splits/Run2/_XTrain_09-06_22-42-46_17.csv')
X17_train = X17_train.iloc[:, 1:]
X17_test = pd.read_csv('Splits/Run2/_XTest_09-06_22-42-46_17.csv')
X17_test = X17_test.iloc[:, 1:]
Y17_train = pd.read_csv('Splits/Run2/_YTrain_09-06_22-42-46_17.csv')
Y17_train = Y17_train.iloc[:, 1:]
Y17_test = pd.read_csv('Splits/Run2/_YTest_09-06_22-42-46_17.csv')
Y17_test = Y17_test.iloc[:, 1:]

XD_train = X17_train[docking_features_only]
XD_test = X17_test[docking_features_only]
models_train,predictionsD_train = clf.fit(XD_train, XD_train, Y17_train, Y17_train)
models_test,predictionsD_test = clf.fit(XD_train, XD_test, Y17_train, Y17_test)
print(predictionsD_test)
        
# Sort models by R-squared in descending order and get the top 3 models
top_3_models = models_test.sort_values(by="R-Squared", ascending=False).head(10)
print(top_3_models)
# Print the top 3 models and their parameters
for model_name in top_3_models.index:
    if model_name == "NuSVR":
        model = clf.models[model_name]
        print(f"Model: {model_name}")
        print(f"Parameters: {model.get_params()}")
        print("-" * 40)
        break

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.svm import NuSVR
import pandas as pd

# Define the feature sets
numeric_features = ['Hbond', 'Hphob', 'VwInt', 'Eintl', 'Dsolv', 'SolEl', 'dTSsc']
categorical_low_features = []  # No categorical_low features in this example
categorical_high_features = []  # No categorical_high features in this example

# Preprocessing pipeline for numeric features
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),  # Impute missing values using the mean
    ('scaler', StandardScaler())  # Standardize the numeric features
])

# Preprocessing pipeline for categorical (low cardinality) features
categorical_low_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),  # Impute missing values with 'missing'
    ('encoding', OneHotEncoder(handle_unknown='ignore', sparse_output=False))  # One-hot encoding
])

# Preprocessing pipeline for categorical (high cardinality) features
categorical_high_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),  # Impute missing values with 'missing'
    ('encoding', OrdinalEncoder())  # Ordinal encoding
])

# Combine all preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_transformer, numeric_features),
        ('categorical_low', categorical_low_transformer, categorical_low_features),
        ('categorical_high', categorical_high_transformer, categorical_high_features)
    ]
)

# Define the NuSVR model with the provided parameters
nusvr_regressor = NuSVR(
    C=1.0,               # Regularization parameter
    nu=0.5,              # Nu parameter
    kernel='rbf',         # Kernel type: Radial Basis Function (RBF)
    gamma='scale',        # Kernel coefficient
    tol=0.001,            # Tolerance for stopping criterion
    shrinking=True,       # Use the shrinking heuristic
    cache_size=200,       # Cache size for kernel computation
    verbose=False         # Suppress verbosity
)

# Create the final pipeline with preprocessing and NuSVR regressor
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', nusvr_regressor)
])

# Fit the model
model.fit(XD_train, Y17_train)
Y_test_values = Y17_test['pIC50'].values

# Predict using the test set
Y_pred = model.predict(XD_test)

# Evaluate and plot actual vs predicted
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score

# Plot the Actual vs Predicted values
plt.figure(figsize=(10, 10))
plt.scatter(Y_test_values, Y_pred, color='blue', label='Predicted vs Actual')

# Perfect fit line
plt.plot([min(Y_test_values), max(Y_test_values)], [min(Y_test_values), max(Y_test_values)], color='red', label='Perfect Fit')

plt.title('Binding Features: Actual vs Predicted using NuSVR', fontsize=20)
plt.xlabel('Actual pIC50 Values', fontsize=20)
plt.ylabel('Predicted pIC50 Values', fontsize=20)
plt.legend(fontsize=20)
# Add R-squared value to the plot
plt.text(0.04, 0.84, f'R-squared = {r2_score(Y_test_values, Y_pred):.2f}', transform=plt.gca().transAxes, fontsize=20, verticalalignment='top')

plt.grid(True)
#plt.legend()
plt.savefig("docking-plot-big.pdf")
plt.grid(True)
plt.show()

# Print performance metrics
print(f"Mean Squared Error: {mean_squared_error(Y_test_values, Y_pred)}")
print(f"R-Squared: {r2_score(Y_test_values, Y_pred)}")

In [ ]:
# PUBLIC COPY: hard-coded private feature identities are redacted placeholders.
#Run directly from saved files

#df_all_features = pd.read_csv('Splits/Run2/allTopFeatures_09-06_22-42-46.csv', header=None)
#f_list = df_all_features.values.tolist()
s_features = ['REDACTED_FEATURE_001','REDACTED_FEATURE_002','REDACTED_FEATURE_021','REDACTED_FEATURE_004','REDACTED_FEATURE_005','REDACTED_FEATURE_020','REDACTED_FEATURE_046','REDACTED_FEATURE_011','REDACTED_FEATURE_012','REDACTED_FEATURE_014','REDACTED_FEATURE_015','REDACTED_FEATURE_047']
docking_features_only = ['Hbond','Hphob','VwInt','Eintl','Dsolv','SolEl','dTSsc']
cmb = s_features + docking_features_only
print(cmb)
clf = LazyRegressor(verbose=0,ignore_warnings=True,custom_metric=None)
#count =0
#for s_features in f_list:

X17_train = pd.read_csv('Splits/Run2/_XTrain_09-06_22-42-46_17.csv')
X17_train = X17_train.iloc[:, 1:]
X17_test = pd.read_csv('Splits/Run2/_XTest_09-06_22-42-46_17.csv')
X17_test = X17_test.iloc[:, 1:]
Y17_train = pd.read_csv('Splits/Run2/_YTrain_09-06_22-42-46_17.csv')
Y17_train = Y17_train.iloc[:, 1:]
Y17_test = pd.read_csv('Splits/Run2/_YTest_09-06_22-42-46_17.csv')
Y17_test = Y17_test.iloc[:, 1:]

XC_train = X17_train[cmb]
XC_test = X17_test[cmb]
models_train,predictionsC_train = clf.fit(XC_train, XC_train, Y17_train, Y17_train)
models_test,predictionsC_test = clf.fit(XC_train, XC_test, Y17_train, Y17_test)
print(predictionsC_test)
        
# Sort models by R-squared in descending order and get the top 3 models
top_3_models = models_test.sort_values(by="R-Squared", ascending=False).head(3)
# Print the top 3 models and their parameters
for model_name in top_3_models.index:
    model = clf.models[model_name]
    print(f"Model: {model_name}")
    print(f"Parameters: {model.get_params()}")
    print("-" * 40)

In [ ]:
# PUBLIC COPY: hard-coded private feature identities are redacted placeholders.
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.svm import NuSVR
import pandas as pd

# Define feature sets
numeric_features = ['REDACTED_FEATURE_001', 'REDACTED_FEATURE_002', 'REDACTED_FEATURE_021', 'REDACTED_FEATURE_004',
                    'REDACTED_FEATURE_005', 'REDACTED_FEATURE_020', 'REDACTED_FEATURE_046', 'REDACTED_FEATURE_011',
                    'REDACTED_FEATURE_012', 'REDACTED_FEATURE_014', 'REDACTED_FEATURE_015', 'REDACTED_FEATURE_047',
                    'Hbond', 'Hphob', 'VwInt', 'Eintl', 'Dsolv', 'SolEl', 'dTSsc']

categorical_low_features = []  # No categorical_low features in this example
categorical_high_features = []  # No categorical_high features in this example

# Preprocessing pipeline for numeric features
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),  # Impute missing values using the mean
    ('scaler', StandardScaler())  # Standardize the numeric features
])

# Preprocessing pipeline for categorical (low cardinality) features
categorical_low_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),  # Impute missing values
    ('encoding', OneHotEncoder(handle_unknown='ignore', sparse_output=False))  # One-hot encoding
])

# Preprocessing pipeline for categorical (high cardinality) features
categorical_high_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),  # Impute missing values
    ('encoding', OrdinalEncoder())  # Ordinal encoding
])

# Combine all preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_transformer, numeric_features),
        ('categorical_low', categorical_low_transformer, categorical_low_features),
        ('categorical_high', categorical_high_transformer, categorical_high_features)
    ]
)

# Define the NuSVR model with the provided parameters
nusvr_regressor = NuSVR(
    C=1.0,               # Regularization parameter
    nu=0.5,              # Nu parameter
    kernel='rbf',         # Kernel type: Radial Basis Function (RBF)
    gamma='scale',        # Kernel coefficient
    tol=0.001,            # Tolerance for stopping criterion
    shrinking=True,       # Use the shrinking heuristic
    cache_size=200,       # Cache size for kernel
    verbose=False         # Suppress verbosity
)

# Create the final pipeline with the preprocessing and NuSVR regressor
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', nusvr_regressor)
])

# Fit the model
model.fit(XC_train, Y17_train)
Y_test_values = Y17_test['pIC50'].values

# Predict using the test set
Y_pred = model.predict(XC_test)

# Evaluate and plot actual vs predicted
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score

# Plot the Actual vs Predicted values
plt.figure(figsize=(10, 10))
plt.scatter(Y_test_values, Y_pred, color='blue', label='Predicted vs Actual')

# Perfect fit line
plt.plot([min(Y_test_values), max(Y_test_values)], [min(Y_test_values), max(Y_test_values)], color='red', label='Perfect Fit')

plt.title('Combined Features: Actual vs Predicted using NuSVR', fontsize=20)
plt.xlabel('Actual pIC50 Values', fontsize=20)
plt.ylabel('Predicted pIC50 Values', fontsize=20)
plt.legend(fontsize=20)
# Add R-squared value to the plot
plt.text(0.04, 0.84, f'R-squared = {r2_score(Y_test_values, Y_pred):.2f}', transform=plt.gca().transAxes, fontsize=20, verticalalignment='top')

plt.grid(True)

plt.savefig("cmb-plot-big.pdf")
plt.grid(True)
plt.show()

# Print performance metrics
print(f"Mean Squared Error: {mean_squared_error(Y_test_values, Y_pred)}")
print(f"R-Squared: {r2_score(Y_test_values, Y_pred)}")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import time

#csv_files = ['new-allQResults_09-06_22-42-46.csv', 'new-allDResults_09-06_22-42-46.csv', 'new-allCResults_09-06_22-42-46.csv']
titles = ['QSAR Features: Violin Plots for Top 10 Models','Binding Features: Violin Plots for Top 10 Models','Combined Features: Violin Plots for Top 10 Models']
df_alls = []
df_3 = pd.read_csv('new-allCResults_09-06_22-42-46.csv', nrows=10, index_col=0)
top_10_models = df_3.index.tolist()
df_1 = pd.read_csv('new-allQResults_09-06_22-42-46.csv', index_col=0)
df_2 = pd.read_csv('new-allDResults_09-06_22-42-46.csv', index_col=0)
df_1_f = df_1.loc[top_10_models]
df_alls.append(df_1_f)
df_2_f = df_2.loc[top_10_models]
df_alls.append(df_2_f)
df_alls.append(df_3)
colors = ['blue', 'orange', 'green']
plt.ioff()

with PdfPages('violin_plots1.pdf') as pdf:
    fig, axes = plt.subplots(3, 1, figsize=(27, 18))

    for i, (df_all, ax) in enumerate(zip(df_alls, axes.flatten())):
        #df_all = pd.read_csv(csv_file, nrows=10)
        df_runs = df_all.drop(columns=['Mean R-SQD', 'Std_Dev'])

        df_transposed = df_runs.T
        df_transposed.columns = top_10_models
        df_transposed = df_transposed.drop(df_transposed.index[0])
        df_transposed.index.name = 'Run'
        #df_transposed.columns = df_transposed.columns.str.replace('Regressor', '', regex=False)

        df_long = df_transposed.reset_index().melt(id_vars='Run', var_name='Model', value_name='R-Squared')

        # Set figure size
        #fig = plt.figure(figsize=(10, 5))

        # Create the violin plot
        sns.violinplot(x='Model', y='R-Squared', data=df_long, palette=[colors[i]], ax=ax)

        ax.set_title(titles[i], fontsize=20)
        ax.set_ylabel('R-Squared Values', fontsize=16)
        ax.set_xlabel('')
        #ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        if(i==2):
            ax.tick_params(axis='x', rotation=30)
            for tick in ax.get_xticklabels():
                tick.set_horizontalalignment('right')
                tick.set_fontsize(16)
        else:
            ax.set_xticklabels([])
        # Set the title and labels
        #plt.title(titles[i], fontsize=20)
        #plt.xlabel('')
        #plt.ylabel('R-Squared', fontsize=16)

        #plt.xticks(rotation=45, ha='right', fontsize=12)

        # Display the plot
plt.tight_layout()
print("Saving the figure....")
#pdf.savefig(fig)  # Save the current figure
fig.savefig('violin_plots_3.pdf')
plt.show()
plt.close()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import time

#csv_files = ['new-allQResults_09-06_22-42-46.csv', 'new-allDResults_09-06_22-42-46.csv', 'new-allCResults_09-06_22-42-46.csv']
titles = ['Binding Features: Violin Plots for Top 10 Models','QSAR Features: Violin Plots for Top 10 Models','Combined Features: Violin Plots for Top 10 Models']

df_c = pd.read_csv('new-allCResults_09-06_22-42-46.csv', nrows=10, index_col=0)
top_10_models = df_c.index.tolist()
df_c_runs = df_c.drop(columns=['Mean R-SQD', 'Std_Dev'])
df_c_transposed = df_c_runs.T
df_c_transposed.columns = top_10_models
df_c_transposed = df_c_transposed.drop(df_c_transposed.index[0])
df_c_transposed.index.name = 'Run'
df_c_long = df_c_transposed.reset_index().melt(id_vars='Run', var_name='Model', value_name='R-Squared')
df_c_long['Source'] = 'Combined Features'

df_q = pd.read_csv('new-allQResults_09-06_22-42-46.csv', index_col=0).loc[top_10_models]
df_q_runs = df_q.drop(columns=['Mean R-SQD', 'Std_Dev'])
df_q_transposed = df_q_runs.T
df_q_transposed.columns = top_10_models
df_q_transposed = df_q_transposed.drop(df_q_transposed.index[0])
df_q_transposed.index.name = 'Run'
df_q_long = df_q_transposed.reset_index().melt(id_vars='Run', var_name='Model', value_name='R-Squared')
df_q_long['Source'] = 'QSAR Features'

df_d = pd.read_csv('new-allDResults_09-06_22-42-46.csv', index_col=0).loc[top_10_models]
df_d_runs = df_d.drop(columns=['Mean R-SQD', 'Std_Dev'])
df_d_transposed = df_d_runs.T
df_d_transposed.columns = top_10_models
df_d_transposed = df_d_transposed.drop(df_d_transposed.index[0])
df_d_transposed.index.name = 'Run'
df_d_long = df_d_transposed.reset_index().melt(id_vars='Run', var_name='Model', value_name='R-Squared')
df_d_long['Source'] = 'Binding Features'

df_combined = pd.concat([df_d_long, df_q_long, df_c_long])
df_combined.rename(columns={'index': 'Model'}, inplace=True)

fig = plt.figure(figsize=(45, 18))
sns.violinplot(x='Model', y='R-Squared', hue='Source', data=df_combined)
plt.xticks(rotation=10, ha='right', fontsize=30)

# Set labels and title
plt.title('Violin Plots for Top 10 Models Across Three Set of Fetures', fontsize=30)
plt.ylabel('R-Squared Values', fontsize=30)
plt.yticks(fontsize=25)
plt.xlabel('')
plt.legend(fontsize=20)

# Show the plot
plt.tight_layout()
plt.show()
fig.savefig('violin_plots_in_one.pdf')
plt.close(fig)


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import time

#csv_files = ['new-allQResults_09-06_22-42-46.csv', 'new-allDResults_09-06_22-42-46.csv', 'new-allCResults_09-06_22-42-46.csv']
titles = ['Binding Features: Violin Plots for Top 5 Models','QSAR Features: Violin Plots for Top 5 Models','Combined Features: Violin Plots for Top 5 Models']

df_c = pd.read_csv('new-allCResults_09-06_22-42-46.csv', nrows=5, index_col=0)
top_10_models = df_c.index.tolist()
df_c_runs = df_c.drop(columns=['Mean R-SQD', 'Std_Dev'])
df_c_transposed = df_c_runs.T
df_c_transposed.columns = top_10_models
df_c_transposed = df_c_transposed.drop(df_c_transposed.index[0])
df_c_transposed.index.name = 'Run'
df_c_long = df_c_transposed.reset_index().melt(id_vars='Run', var_name='Model', value_name='R-Squared')
df_c_long['Source'] = 'Combined Features'

df_q = pd.read_csv('new-allQResults_09-06_22-42-46.csv', index_col=0).loc[top_10_models]
df_q_runs = df_q.drop(columns=['Mean R-SQD', 'Std_Dev'])
df_q_transposed = df_q_runs.T
df_q_transposed.columns = top_10_models
df_q_transposed = df_q_transposed.drop(df_q_transposed.index[0])
df_q_transposed.index.name = 'Run'
df_q_long = df_q_transposed.reset_index().melt(id_vars='Run', var_name='Model', value_name='R-Squared')
df_q_long['Source'] = 'QSAR Features'

df_d = pd.read_csv('new-allDResults_09-06_22-42-46.csv', index_col=0).loc[top_10_models]
df_d_runs = df_d.drop(columns=['Mean R-SQD', 'Std_Dev'])
df_d_transposed = df_d_runs.T
df_d_transposed.columns = top_10_models
df_d_transposed = df_d_transposed.drop(df_d_transposed.index[0])
df_d_transposed.index.name = 'Run'
df_d_long = df_d_transposed.reset_index().melt(id_vars='Run', var_name='Model', value_name='R-Squared')
df_d_long['Source'] = 'Binding Features'

df_combined = pd.concat([df_d_long, df_q_long, df_c_long])
df_combined.rename(columns={'index': 'Model'}, inplace=True)

fig = plt.figure(figsize=(45, 18))
sns.violinplot(x='Model', y='R-Squared', hue='Source', data=df_combined, bw_method=1.0, density_norm="width")
plt.xticks(rotation=10, ha='right', fontsize=60)

# Set labels and title
#plt.title('Violin Plots for Top 5 Models Across Three Set of Features', fontsize=60)
plt.ylabel('R-Squared Values', fontsize=60)
plt.yticks(fontsize=60)
plt.xlabel('')
plt.legend(fontsize=60)

# Show the plot
plt.tight_layout()
plt.show()
fig.savefig('violin_plots_in_one-big.pdf')
plt.close(fig)


In [ ]:
predictionsQ_test

In [ ]:
predictionsQ_test_file = predictionsQ_test.reset_index()
filename = f'predictionsQ_test_{current_time}.csv'
predictionsQ_test_file.to_csv(os.path.join(subdirectory, filename), index=False)

In [ ]:
predictionsQ_test

In [ ]:
predictionsQ_train_aligned = predictionsQ_train.reindex(predictionsQ_test.index)
predictionsQ_train_aligned

In [ ]:
predictionsQ_train_file = predictionsQ_train_aligned.reset_index()
filename = f'predictionsQ_train_{current_time}.csv'
predictionsQ_train_file.to_csv(os.path.join(subdirectory, filename), index=False)

In [ ]:
X_final = X_reduced[s2]
print(X_final.columns)

In [ ]:
# PUBLIC COPY: hard-coded private feature identities are redacted placeholders.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.svm import SVR

# Define the numeric and categorical features (as per the output)
numeric_features = ['REDACTED_FEATURE_001', 'REDACTED_FEATURE_004', 'REDACTED_FEATURE_015', 'REDACTED_FEATURE_005',
                    'REDACTED_FEATURE_006', 'REDACTED_FEATURE_007', 'REDACTED_FEATURE_009', 'REDACTED_FEATURE_010',
                    'REDACTED_FEATURE_011', 'REDACTED_FEATURE_019', 'REDACTED_FEATURE_012']

categorical_low_features = []  # Replace with your categorical features if any
categorical_high_features = []  # Replace with your categorical features if any

X_final = X_reduced[numeric_features]
background = shap.sample(X_final, 100)

# Create the preprocessing pipelines for both numeric and categorical features
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_low_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoding', OneHotEncoder(handle_unknown='ignore', sparse=False))
])

categorical_high_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoding', OrdinalEncoder())
])

# Combine the preprocessing pipelines into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_transformer, numeric_features),
        ('categorical_low', categorical_low_transformer, categorical_low_features),
        ('categorical_high', categorical_high_transformer, categorical_high_features)
    ]
)

# Define the SVR model with the parameters provided
svr_model = SVR(C=1.0, kernel='rbf', gamma='scale', epsilon=0.1, shrinking=True, tol=0.001, verbose=False)

# Create the complete pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', svr_model)
])

# Now, you can fit the pipeline to your data
# Assuming X and Y are your feature matrix and target variable respectively
svr_model.fit(X_final, Y)
background = shap.sample(X_final, 100)

# Step 2: Initialize SHAP TreeExplainer
explainer = shap.KernelExplainer(svr_model.predict, background)

# Step 3: Calculate SHAP values
shap_values = explainer.shap_values(X_final)

# Step 4: Create SHAP Plots
# Summary plot to visualize feature importance
shap.summary_plot(shap_values, X_final, plot_type="bar")

# Detailed feature impact plot
shap.summary_plot(shap_values, X_final)

In [ ]:
# PUBLIC COPY: hard-coded private feature identities are redacted placeholders.
#SHAP
import shap
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingRegressor

# Step 1: Define the numeric and categorical features (from the output)
numeric_features = ['REDACTED_FEATURE_001', 'REDACTED_FEATURE_002', 'REDACTED_FEATURE_004', 'REDACTED_FEATURE_003', 'REDACTED_FEATURE_013', 'REDACTED_FEATURE_014', 'REDACTED_FEATURE_015', 'REDACTED_FEATURE_016', 'REDACTED_FEATURE_005', 'REDACTED_FEATURE_006', 'REDACTED_FEATURE_007', 'REDACTED_FEATURE_009', 'REDACTED_FEATURE_010', 'REDACTED_FEATURE_017', 'REDACTED_FEATURE_018', 'REDACTED_FEATURE_011', 'REDACTED_FEATURE_019', 'REDACTED_FEATURE_012']
categorical_low_features = []  # Replace with your categorical features if any
categorical_high_features = []  # Replace with your categorical features if any
X_final = X_reduced[numeric_features]

# Step 2: Create the preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_low_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoding', OneHotEncoder(handle_unknown='ignore', sparse=False))
])

categorical_high_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoding', OrdinalEncoder())
])

# Combine transformers into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_transformer, numeric_features),
        ('categorical_low', categorical_low_transformer, categorical_low_features),
        ('categorical_high', categorical_high_transformer, categorical_high_features)
    ]
)

# Step 3: Create the model pipeline
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', HistGradientBoostingRegressor(
        random_state=42,
        categorical_features=None,
        early_stopping='auto',
        learning_rate=0.1,
        loss='squared_error',
        max_bins=255,
        max_depth=None,
        max_iter=100,
        max_leaf_nodes=31,
        min_samples_leaf=20,
        n_iter_no_change=10,
        validation_fraction=0.1,
        warm_start=False
    ))
])

# Step 4: Train the model on your final dataset
model.fit(X_final, Y)

trained_model = model.named_steps['regressor']
# Step 2: Initialize SHAP TreeExplainer
explainer = shap.TreeExplainer(trained_model)

# Step 3: Calculate SHAP values
shap_values = explainer.shap_values(X_final)

# Step 4: Create SHAP Plots
# Summary plot to visualize feature importance
shap.summary_plot(shap_values, X_final, plot_type="bar")

# Detailed feature impact plot
shap.summary_plot(shap_values, X_final)

In [ ]:
# PUBLIC COPY: hard-coded private feature identities are redacted placeholders.
#SHAP
import shap
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingRegressor

# Step 1: Define the numeric and categorical features (from the output)
numeric_features = ['REDACTED_FEATURE_001', 'REDACTED_FEATURE_004', 'REDACTED_FEATURE_005', 'REDACTED_FEATURE_006',
                    'REDACTED_FEATURE_007', 'REDACTED_FEATURE_009', 'REDACTED_FEATURE_010', 'REDACTED_FEATURE_011',
                    'REDACTED_FEATURE_019', 'REDACTED_FEATURE_012']
categorical_low_features = []  # Replace with your categorical features if any
categorical_high_features = []  # Replace with your categorical features if any


X_final = X_reduced[numeric_features]
print(X_final.columns)

# Step 2: Create the preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_low_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoding', OneHotEncoder(handle_unknown='ignore', sparse=False))
])

categorical_high_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoding', OrdinalEncoder())
])

# Combine transformers into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_transformer, numeric_features),
        ('categorical_low', categorical_low_transformer, categorical_low_features),
        ('categorical_high', categorical_high_transformer, categorical_high_features)
    ]
)

# Step 3: Create the model pipeline
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', HistGradientBoostingRegressor(
        random_state=42,
        categorical_features=None,
        early_stopping='auto',
        learning_rate=0.1,
        loss='squared_error',
        max_bins=255,
        max_depth=None,
        max_iter=100,
        max_leaf_nodes=31,
        min_samples_leaf=20,
        n_iter_no_change=10,
        validation_fraction=0.1,
        warm_start=False
    ))
])

# Step 4: Train the model on your final dataset
model.fit(X_final, Y)

trained_model = model.named_steps['regressor']
# Step 2: Initialize SHAP TreeExplainer
explainer = shap.TreeExplainer(trained_model)

# Step 3: Calculate SHAP values
shap_values = explainer.shap_values(X_final)

# Step 4: Create SHAP Plots
# Summary plot to visualize feature importance
plt.figure()
shap.summary_plot(shap_values, X_final, plot_type="bar", show=False)
plt.savefig('shap_summary_bar08271.png', bbox_inches='tight', dpi=300)
plt.close()

# Detailed feature impact plot
plt.figure()
shap.summary_plot(shap_values, X_final, show=False)
plt.savefig('shap_summary_detailed08271.png', bbox_inches='tight', dpi=300)
plt.close()

In [ ]:
# PUBLIC COPY: hard-coded private feature identities are redacted placeholders.
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.ensemble import GradientBoostingRegressor

# Define the numeric and categorical features based on your provided columns
numeric_features = [
    'REDACTED_FEATURE_001', 'REDACTED_FEATURE_004', 'REDACTED_FEATURE_015', 'REDACTED_FEATURE_005',
    'REDACTED_FEATURE_006', 'REDACTED_FEATURE_007', 'REDACTED_FEATURE_009', 'REDACTED_FEATURE_010',
    'REDACTED_FEATURE_011', 'REDACTED_FEATURE_019', 'REDACTED_FEATURE_012'
]

categorical_low_features = []  # Add any low-cardinality categorical features here
categorical_high_features = []  # Add any high-cardinality categorical features here

# Define the preprocessor for numerical and categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', Pipeline(steps=[
            ('imputer', SimpleImputer()),  # Handle missing values
            ('scaler', StandardScaler())  # Standardize the numeric data
        ]), numeric_features),
        
        ('categorical_low', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('encoding', OneHotEncoder(handle_unknown='ignore', sparse=False))
        ]), categorical_low_features),
        
        ('categorical_high', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('encoding', OrdinalEncoder())
        ]), categorical_high_features)
    ]
)

# Define the final pipeline including the preprocessor and the regressor
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', GradientBoostingRegressor(
        random_state=42,  # Set the random state for reproducibility
        alpha=0.9,
        criterion='friedman_mse',
        learning_rate=0.1,
        loss='squared_error',
        max_depth=3,
        min_samples_leaf=1,
        min_samples_split=2,
        n_estimators=100,
        subsample=1.0,
        tol=0.0001,
        validation_fraction=0.1
    ))
])

pipeline.fit(XQ_train, Y_train)

Y_pred = pipeline.predict(XQ_test)
# Calculate Mean Squared Error (optional)
mse = mean_squared_error(Y_test, Y_pred)
print(f"Mean Squared Error: {mse}")

# Calculate R-squared value
r_squared = r2_score(Y_test, Y_pred)

# Plotting Actual vs Predicted
plt.figure(figsize=(10, 10))
plt.scatter(Y_test, Y_pred, color='blue', label='Predicted vs Actual')
plt.plot([min(Y_test), max(Y_test)], [min(Y_test), max(Y_test)], color='red', lw=2, label='Ideal Fit')

# Add R-squared value to the plot
plt.text(0.05, 0.95, f'R-squared = {r_squared:.2f}', transform=plt.gca().transAxes, fontsize=12, verticalalignment='top')

plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title(f'Actual vs Predicted using GradientBoostingRegressor')
#plt.legend()
plt.grid(True)
plt.savefig("qsarplot.png")
plt.show()


In [ ]:
# PUBLIC COPY: hard-coded private feature identities are redacted placeholders.
from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
import pandas as pd

# Define your features based on the printout
numeric_features = ['REDACTED_FEATURE_001', 'REDACTED_FEATURE_002', 'REDACTED_FEATURE_004', 'REDACTED_FEATURE_013', 'REDACTED_FEATURE_048']
categorical_low_features = []  # Add any low-cardinality categorical features here
categorical_high_features = []  # Add any high-cardinality categorical features here

# Preprocessor setup
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler())]), numeric_features),
        ('categorical_low', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('encoding', OneHotEncoder(handle_unknown='ignore', sparse=False))]), categorical_low_features),
        ('categorical_high', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('encoding', OrdinalEncoder())]), categorical_high_features)
    ]
)

# LGBMRegressor model
lgbm = LGBMRegressor(random_state=42)

# Full pipeline
model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('regressor', lgbm)])

# Fit the model
model_pipeline.fit(XQ_train, Y_train)

# Make predictions
Y_pred = model_pipeline.predict(XQ_test)

# Calculate Mean Squared Error (optional)
mse = mean_squared_error(Y_test, Y_pred)
print(f"Mean Squared Error: {mse}")

# Calculate R-squared value
r_squared = r2_score(Y_test, Y_pred)


# Plotting Actual vs Predicted
plt.figure(figsize=(10, 10))
plt.scatter(Y_test, Y_pred, color='blue', label='Predicted vs Actual')
plt.plot([min(Y_test), max(Y_test)], [min(Y_test), max(Y_test)], color='red', lw=2, label='Ideal Fit')

# Add R-squared value to the plot
plt.text(0.05, 0.95, f'R-squared = {r_squared:.2f}', transform=plt.gca().transAxes, fontsize=12, verticalalignment='top')

plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title(f'Actual vs Predicted using LGBMRegressor')
#plt.legend()
plt.grid(True)
plt.show()

In [ ]:
s_list = s.tolist()
docking_features_only_list = list(docking_features_only)
cmb_features = s_list + docking_features_only_list
print(cmb_features)

In [ ]:
XC_train = X_train[cmb_features]
XC_test = X_test[cmb_features]
clf = LazyRegressor(verbose=0,ignore_warnings=True, custom_metric=None)
models_train,predictionsC_train = clf.fit(XC_train, XC_train, Y_train, Y_train)
models_test,predictionsC_test = clf.fit(XC_train, XC_test, Y_train, Y_test)

# Sort models by R-squared in descending order and get the top 3 models
top_3_models = models_test.sort_values(by="R-Squared", ascending=False).head(3)

# Print the top 3 models and their parameters
for model_name in top_3_models.index:
    model = clf.models[model_name]
    print(f"Model: {model_name}")
    print(f"Parameters: {model.get_params()}")
    print("-" * 40)

In [ ]:
predictionsC_test

In [ ]:
predictionsC_test_file = predictionsC_test.reset_index()
filename = f'predictionsC_test_{current_time}.csv'
predictionsC_test_file.to_csv(os.path.join(subdirectory, filename), index=False)

In [ ]:
predictionsC_train_aligned = predictionsC_train.reindex(predictionsC_test.index)
predictionsC_train_aligned

In [ ]:
predictionsC_train_file = predictionsC_train_aligned.reset_index()
filename = f'predictionsC_train_{current_time}.csv'
predictionsC_train_file.to_csv(os.path.join(subdirectory, filename), index=False)

In [ ]:
# PUBLIC COPY: hard-coded private feature identities are redacted placeholders.
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.svm import NuSVR

# Define the preprocessing steps for numerical and categorical data
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer()),  # Handle missing values
    ('scaler', StandardScaler())   # Standardize features
])

categorical_low_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),  # Handle missing values
    ('encoding', OneHotEncoder(handle_unknown='ignore', sparse=False))  # OneHotEncode categorical features
])

categorical_high_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),  # Handle missing values
    ('encoding', OrdinalEncoder())  # OrdinalEncode categorical features
])

# Combine the numerical and categorical transformers into a ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('numeric', numeric_transformer, ['REDACTED_FEATURE_001', 'REDACTED_FEATURE_015', 'REDACTED_FEATURE_005', 'REDACTED_FEATURE_006',
                                      'REDACTED_FEATURE_007', 'REDACTED_FEATURE_009', 'REDACTED_FEATURE_010', 'REDACTED_FEATURE_011',
                                      'REDACTED_FEATURE_019', 'REDACTED_FEATURE_012', 'Hbond', 'Hphob', 'VwInt', 'Eintl',
                                      'Dsolv', 'SolEl', 'dTSsc']),
    ('categorical_low', categorical_low_transformer, []),  # Replace with actual low cardinality categorical features if available
    ('categorical_high', categorical_high_transformer, [])  # Replace with actual high cardinality categorical features if available
])

# Define the model
model = NuSVR(C=1.0, cache_size=200, coef0=0.0, degree=3, gamma='scale', kernel='rbf', 
              max_iter=-1, nu=0.5, shrinking=True, tol=0.001, verbose=False)

# Create the pipeline
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('regressor', model)])

# Fit the pipeline to the training data
pipeline.fit(XC_train, Y_train)

# Make predictions on the test data
Y_pred = pipeline.predict(XC_test)

# Calculate Mean Squared Error (optional)
mse = mean_squared_error(Y_test, Y_pred)
print(f"Mean Squared Error: {mse}")

# Calculate R-squared value
r_squared = r2_score(Y_test, Y_pred)


# Plotting Actual vs Predicted
plt.figure(figsize=(10, 10))
plt.scatter(Y_test, Y_pred, color='blue', label='Predicted vs Actual')
plt.plot([min(Y_test), max(Y_test)], [min(Y_test), max(Y_test)], color='red', lw=2, label='Ideal Fit')

# Add R-squared value to the plot
plt.text(0.05, 0.95, f'R-squared = {r_squared:.2f}', transform=plt.gca().transAxes, fontsize=12, verticalalignment='top')

plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title(f'Actual vs Predicted using NuSVR')
#plt.legend()
plt.grid(True)
plt.savefig('cmb.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# PUBLIC COPY: hard-coded private feature identities are redacted placeholders.
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesRegressor

# Assuming the columns are named as shown in your output:
numeric_features = ['REDACTED_FEATURE_001', 'REDACTED_FEATURE_002', 'REDACTED_FEATURE_004', 'REDACTED_FEATURE_013', 'REDACTED_FEATURE_048', 'Hbond', 'Hphob', 'VwInt', 'Eintl', 'Dsolv', 'SolEl', 'dTSsc']
categorical_low_features = []  # Add any low-cardinality categorical features here
categorical_high_features = []  # Add any high-cardinality categorical features here

# Preprocessor setup
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler())]), numeric_features),
        ('categorical_low', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('encoding', OneHotEncoder(handle_unknown='ignore', sparse=False))]), categorical_low_features),
        ('categorical_high', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('encoding', OrdinalEncoder())]), categorical_high_features)
    ]
)

# SVR model
svr = SVR(C=1.0, kernel='rbf', gamma='scale', epsilon=0.1)

# Full pipeline
model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('regressor', svr)])

# Now you can fit this pipeline on your data
model_pipeline.fit(XC_train, Y_train)

# And make predictions
Y_pred = model_pipeline.predict(XC_test)

# Calculate Mean Squared Error (optional)
mse = mean_squared_error(Y_test, Y_pred)
print(f"Mean Squared Error: {mse}")

# Calculate R-squared value
r_squared = r2_score(Y_test, Y_pred)


# Plotting Actual vs Predicted
plt.figure(figsize=(10, 10))
plt.scatter(Y_test, Y_pred, color='blue', label='Predicted vs Actual')
plt.plot([min(Y_test), max(Y_test)], [min(Y_test), max(Y_test)], color='red', lw=2, label='Ideal Fit')

# Add R-squared value to the plot
plt.text(0.05, 0.95, f'R-squared = {r_squared:.2f}', transform=plt.gca().transAxes, fontsize=12, verticalalignment='top')

plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title(f'Actual vs Predicted using SVR')
#plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns

dfv = pd.read_csv('output_file3.csv') # read in QSAR only dataset
dfv_sorted = dfv.sort_values(by='pIC50', ascending=False)

dfp = dfv_sorted[selected_features.union(['pIC50'])]
print(selected_features)
print(dfp)

In [ ]:
#Lets try bipartite Graph
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd

# Step 1: Load your data
# Assuming your dataset is in a DataFrame called df, where the last column is the pIC50 target
# and the first 156 columns are the binary predictors.


# Sample structure for illustrative purposes
# df = pd.DataFrame({
#     'predictor1': [0, 1, 0, 1],
#     'predictor2': [1, 0, 1, 1],
#     ...
#     'pIC50': [5.6, 7.8, 6.1, 4.5]
# })

# Step 2: Binarize the pIC50 if necessary (or create categories)
# You can categorize pIC50 values, but if you want to keep them as continuous values, you can skip this step.
dfp['pIC50_category'] = pd.qcut(dfn['pIC50'], q=15, labels=False)
dfp['pIC50_category'] = dfp['pIC50_category'] + 1

# Step 3: Create the bipartite graph
B = nx.Graph()

# Add pIC50 nodes
pIC50_nodes = dfp['pIC50_category'].unique()  # You can use categorized pIC50 if you binned it
B.add_nodes_from(pIC50_nodes, bipartite=0)

# Add predictor nodes
predictors = dfp.columns[:-2]  # All columns except the last one
print(predictors)
B.add_nodes_from(predictors, bipartite=1)


# Step 4: Add edges between predictors and pIC50 values
for index, row in dfp.iterrows():
    for predictor in predictors:
        if row[predictor] == 1:
            B.add_edge(row['pIC50_category'], predictor)  # Use row['pIC50_category'] if categorized
            

edges = list(B.edges())

# Step 2: Convert the list of edges to a DataFrame
edges_df = pd.DataFrame(edges, columns=['source', 'target'])
edges_list = edges_df.values.tolist()
print(edges_list)

# Step 3: Export the DataFrame to a CSV file
edges_df.to_csv('edges.csv', index=False)
# Step 5: Draw the bipartite graph
plt.figure(figsize=(14, 10))

# Define the layout for bipartite graphs
pos = nx.bipartite_layout(B, pIC50_nodes)

# Draw nodes and edges with colors
nx.draw_networkx_nodes(B, pos, nodelist=pIC50_nodes, node_color='lightgreen', node_size=500, alpha=0.8, label='pIC50 Categories')
nx.draw_networkx_nodes(B, pos, nodelist=predictors, node_color='lightblue', node_size=500, alpha=0.8, label='Predictors')
nx.draw_networkx_edges(B, pos, width=1.0, alpha=0.5)
nx.draw_networkx_labels(B, pos, font_size=10)

plt.title("Bipartite Graph of Predictors and pIC50")
plt.legend()
plt.show()

In [ ]:
import pandas as pd

# Step 1: Read the CSV file into a DataFrame
edges_df = pd.read_csv('edges.csv')

# Step 2: Convert the DataFrame to a list of lists
edges_list = edges_df.values.tolist()

# Output the list to check
print(edges_list)

In [ ]:
# Calculate Cohen's D value for each predictor
# Function to calculate Cohen's d
def cohen_d(group1, group2):
    diff = group1.mean() - group2.mean()
    pooled_std = np.sqrt(((group1.size - 1) * group1.std()**2 + (group2.size - 1) * group2.std()**2) / (group1.size + group2.size - 2))
    return diff / pooled_std

cohen_d_values = {}

for predictor in XQ_train.columns:  # Iterate over each predictor
    group1 = YQ_train[XQ_train[predictor] == 0]
    group2 = YQ_train[XQ_train[predictor] == 1]
    
    d_value = cohen_d(group1, group2)
    cohen_d_values[predictor] = d_value

# Display the Cohen's d values
print(cohen_d_values)

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
selected_df = X[selected_features]
# Create violin plots for each predictor
with PdfPages('violin_plots_best_10.pdf') as pdf:
    for pred in selected_features:
        plt.figure(figsize=(10, 6))
        sns.violinplot(x=pred, y=Y, data=selected_df)
        plt.title(f'Violin Plot for {pred}')
        plt.xlabel(pred)
        plt.ylabel('Pic50')
        
        #plt.text(0.95, 0.95, f'p-value: {p_value:.3e}', 
                 #horizontalalignment='right', 
                 #verticalalignment='center', 
                 #transform=plt.gca().transAxes,
                 #fontsize=12, 
                 #bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'))
        
        d_value = cohen_d_values[pred]
        plt.text(0.95, 0.90, f"Cohen's d: {d_value:.2f}", 
                 horizontalalignment='right', 
                 verticalalignment='center', 
                 transform=plt.gca().transAxes,
                 fontsize=12, 
                 bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'))
        
        pdf.savefig()
        plt.close()

In [ ]:
# Performance table of the training set (50% subset)
predictions_train.to_csv('predictions_train_cmb_10.csv', index=True)
predictions_train

In [ ]:
# Performance table of the test set (50% subset)
predictions_test.to_csv('predictions_test_cmb_10.csv', index=True)
predictions_test

Data visualization of model performance.

In [ ]:
# Bar plot of R-squared values
import matplotlib.pyplot as plt
import seaborn as sns

#train["R-Squared"] = [0 if i < 0 else i for i in train.iloc[:,0] ]

plt.figure(figsize=(5, 10))
sns.set_theme(style="whitegrid")
ax = sns.barplot(y=predictions_train.index, x="R-Squared", data=predictions_train)
ax.set(xlim=(0, 1))

In [ ]:
# Bar plot of RMSE values
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(5, 10))
sns.set_theme(style="whitegrid")
ax = sns.barplot(y=predictions_train.index, x="RMSE", data=predictions_train)
ax.set(xlim=(0, 10))

In [ ]:
# Bar plot of calculation time
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(5, 10))
sns.set_theme(style="whitegrid")
ax = sns.barplot(y=predictions_train.index, x="Time Taken", data=predictions_train)
ax.set(xlim=(0, 10))

## Synthetic visualization previews

These illustrations were added for this public code sample using randomly generated values. They contain no lab records or measured model results; no models were trained to produce them. They demonstrate plotting styles, not research findings. The previews and their generator were prepared with AI assistance.

![Synthetic fingerprint heatmap](../assets/synthetic_fingerprint_heatmap.png)

![Synthetic model-comparison violin plots](../assets/synthetic_model_violin.png)
